# Controlled RAG Evaluation for University Policy Documents

Public portfolio version of the **final executed CN7000 dissertation pipeline** used to compare an LLM-only baseline, Basic RAG and Enhanced RAG on a fixed university-policy corpus.

The completed experiment used **15 version-controlled public documents**, **120 researcher-verified questions** (30 development / 90 protected held-out), **143 stable evidence units**, and produced **270/270 successful held-out system outputs**.

> **Portfolio cleanup:** stored Colab outputs and viva/submission-specific notes have been removed from this public copy. The experimental logic is preserved. The Hugging Face model revisions below are pinned to the exact immutable commits recorded in the frozen final run.

> **Use note:** this is a research evaluation, not an official source of university policy advice.


## Research and evaluation safeguards

- The corpus is fixed to the 15 supplied documents; the notebook never downloads newer web versions.
- Development questions are used for retrieval choices. Held-out questions are not used to choose settings.
- The generator model, prompts, seed, decoding rule and top-*k* remain fixed across systems.
- The Basic RAG configuration is the pre-specified 512-token, 10% overlap, no-reranking baseline.
- Development ablations change one retrieval factor at a time. A required unscored T4 sanity sample then confirms that all three final systems run before the Enhanced configuration and protocol are frozen.
- There is no independent reviewer component. Manual evaluation is a researcher-only, system-blinded process, consistent with the agreed ethics decision.
- An empty model output is scored as incorrect. Only a confirmed infrastructure failure permits one logged rerun; a second failure remains missing.
- Unequal category sizes are reported with within-category percentages and a category macro-average as well as the ordinary overall score.

The implementation below is newly structured for this dissertation. General workflow ideas from teaching exercises and the author's earlier practice are not reproduced verbatim. The methodology should disclose that AI and Python assisted preparation and code development, while the researcher manually verified the final dataset against the frozen sources.


## 1. Install the reproducible runtime

The environment and installed package versions are recorded in the frozen protocol. A Colab restart may be requested after installation if Colab has already loaded an incompatible binary package.

When resuming a frozen project, the later protocol check compares the current Python, PyTorch, CUDA, GPU and package versions with the recorded environment and stops on any mismatch.


In [ ]:
import subprocess
import sys

REQUIRED_PACKAGES = [
    "pymupdf>=1.24,<1.27",
    "python-docx>=1.1,<1.3",
    "pandas>=2.1,<2.4",
    "pyarrow>=15,<22",
    "sentence-transformers>=3.0,<6.0",
    "transformers>=4.45,<4.57",
    "accelerate>=1.0,<2.0",
    "bitsandbytes>=0.43,<0.49",
    "faiss-cpu>=1.8,<1.13",
    "scipy>=1.11,<1.17",
    "statsmodels>=0.14,<0.15",
    "seaborn>=0.13,<0.14",
    "openpyxl>=3.1,<4.0",
    "tqdm>=4.66,<5.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES])
print("Dependencies installed. Continue to the next cell.")


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import platform
import random
import re
import shutil
import statistics
import time
import unicodedata
import uuid
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Optional

import faiss
import fitz
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import torch
from docx import Document
from docx.document import Document as DocumentType
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import Table
from docx.text.paragraph import Paragraph
from sentence_transformers import CrossEncoder, SentenceTransformer
from huggingface_hub import HfApi
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 7023
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

pd.set_option("display.max_colwidth", 160)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Select or upload the simplified dataset

The preferred input is one folder (or ZIP) containing:

1. `15_Frozen_Documents/` with exactly 15 PDF/DOCX files;
2. the final reviewed questions CSV;
3. the final reviewed evidence-units CSV without offsets; and
4. the simplified corpus manifest without derived hashes or character counts.

The **Drive** option asks for an existing folder or ZIP in My Drive. The **Upload** option opens Colab's file picker and stores the uploaded material inside the new project folder in Drive. In both cases the source is copied into a project input snapshot; the original is not modified.

The reviewed input ZIP supplied for this project is compatible with this schema. CSV files are identified by their required column names rather than by an exact filename, so renaming the ZIP after download does not change the experiment.


In [ ]:
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False


def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if target != destination_resolved and destination_resolved not in target.parents:
                raise ValueError(f"Unsafe ZIP member rejected: {item.filename}")
        archive.extractall(destination)


def copy_input_snapshot(source: Path, destination: Path) -> Path:
    if any(destination.iterdir()) if destination.exists() else False:
        raise FileExistsError(
            f"The new input snapshot is not empty: {destination}. "
            "Choose a new project folder so earlier frozen work is not overwritten."
        )
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        target = destination / source.name
        shutil.copytree(source, target)
        return target
    if source.suffix.lower() == ".zip":
        copied_zip = destination / source.name
        shutil.copy2(source, copied_zip)
        extracted = destination / "extracted"
        safe_extract_zip(copied_zip, extracted)
        return extracted
    shutil.copy2(source, destination / source.name)
    return destination


if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path.cwd()
    print("Not running in Colab; local paths will be used for a structural test.")

default_name = f"UEL_RAG_Dissertation_{datetime.now().strftime('%Y%m%d')}"
project_name = input(f"New project folder name [{default_name}]: ").strip() or default_name
if Path(project_name).name != project_name or project_name in {".", ".."}:
    raise ValueError("Enter a single new folder name, not a path.")
PROJECT_ROOT = DRIVE_ROOT / project_name
RESUME_PROJECT = False
if PROJECT_ROOT.exists():
    if any(PROJECT_ROOT.iterdir()):
        resume_confirmation = input(
            f"{PROJECT_ROOT} already contains a project. Type RESUME to verify and continue it, "
            "or press Enter to stop: "
        ).strip()
        if resume_confirmation != "RESUME":
            raise FileExistsError("Existing project was not resumed. Choose a new project folder name.")
        RESUME_PROJECT = True
else:
    PROJECT_ROOT.mkdir(parents=True, exist_ok=False)

DIRS = {
    "root": PROJECT_ROOT,
    "input": PROJECT_ROOT / "00_input_snapshot",
    "build": PROJECT_ROOT / "01_preprocessing_build",
    "frozen": PROJECT_ROOT / "02_frozen_dataset",
    "development": PROJECT_ROOT / "03_development",
    "protocol": PROJECT_ROOT / "04_frozen_protocol",
    "runs": PROJECT_ROOT / "05_model_runs",
    "evaluation": PROJECT_ROOT / "06_evaluation",
    "figures": PROJECT_ROOT / "07_figures",
    "exports": PROJECT_ROOT / "08_exports",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

if RESUME_PROJECT:
    STAGED_INPUT_ROOT = DIRS["input"]
    if not STAGED_INPUT_ROOT.exists() or not any(STAGED_INPUT_ROOT.iterdir()):
        raise FileNotFoundError("The resumed project has no input snapshot.")
    print("Resuming the existing project; the saved input snapshot will be rediscovered and verified.")
else:
    mode = input("Input mode — type DRIVE or UPLOAD [DRIVE]: ").strip().lower() or "drive"
    if mode == "drive":
        raw_path = input(
            "Folder or ZIP path in My Drive (absolute, or relative to MyDrive): "
        ).strip()
        if not raw_path:
            raise ValueError("A Drive input path is required.")
        source = Path(raw_path)
        if not source.is_absolute():
            source = DRIVE_ROOT / source
        if not source.exists():
            raise FileNotFoundError(source)
        STAGED_INPUT_ROOT = copy_input_snapshot(source, DIRS["input"])
    elif mode == "upload":
        if not IN_COLAB:
            raise RuntimeError("UPLOAD mode requires Google Colab; use DRIVE with a local path here.")
        print("Upload the dataset ZIP, or the three CSVs plus the 15 document files.")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No files were uploaded.")
        upload_folder = DIRS["input"] / "manual_upload"
        upload_folder.mkdir(parents=True, exist_ok=False)
        for name, payload in uploaded.items():
            target = upload_folder / Path(name).name
            target.write_bytes(payload)
        zip_files = list(upload_folder.glob("*.zip"))
        if len(zip_files) == 1:
            extracted = upload_folder / "extracted"
            safe_extract_zip(zip_files[0], extracted)
            STAGED_INPUT_ROOT = extracted
        elif not zip_files:
            STAGED_INPUT_ROOT = upload_folder
        else:
            raise ValueError("Upload one dataset ZIP only, or upload the loose input files without ZIPs.")
    else:
        raise ValueError("Input mode must be DRIVE or UPLOAD.")

print("Project folder:", PROJECT_ROOT)
print("Input snapshot:", STAGED_INPUT_ROOT)


In [ ]:
@dataclass(frozen=True)
class Settings:
    project_name: str = "UEL_RAG_CONTROLLED_EVALUATION"
    corpus_version: str = "UEL_PUBLIC_POLICIES_2025_26_V1"
    question_dataset_version: str = "UEL_QA_120_FINAL_REVIEWED_V1"
    seed: int = SEED

    generator_model: str = "Qwen/Qwen2.5-3B-Instruct"
    embedding_model: str = "BAAI/bge-small-en-v1.5"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L6-v2"
    model_revision_ref: str = "main"  # Resolved once to immutable commit SHAs before model loading.
    query_instruction: str = "Represent this sentence for searching relevant passages: "

    top_k: int = 5
    basic_chunk_size: int = 512
    basic_overlap_pct: int = 10
    chunk_sizes: tuple[int, ...] = (256, 384, 512)
    overlap_pcts: tuple[int, ...] = (0, 10, 20)
    candidate_pool_options: tuple[int, ...] = (10, 15, 20)
    default_candidate_pool: int = 15

    max_input_tokens: int = 4096
    max_new_tokens: int = 256
    generation_do_sample: bool = False
    generation_temperature: Optional[float] = None
    generation_num_beams: int = 1
    embedding_batch_size: int = 64
    evidence_overlap_threshold: float = 0.80
    bootstrap_repetitions: int = 10_000
    expected_runtime: str = "Google Colab Python 3 with NVIDIA T4 GPU"


CFG = Settings()

REFUSAL_TEXT = "The provided university documents do not contain enough information to answer this question."
RAG_PROMPT = """Answer the question using only the evidence passages below. Give a concise answer and cite the relevant document and passage identifier in square brackets after each factual statement. If the passages are insufficient, reply exactly: '{refusal}' Do not guess, use outside knowledge, or invent a source.

Question:
{question}

Evidence passages:
{evidence}

Answer:"""
LLM_ONLY_PROMPT = """Answer the question concisely using only information you can reliably verify from your existing model knowledge. Do not invent a source. If you cannot verify the requested UEL-specific information, state that you cannot verify the answer.

Question:
{question}

Answer:"""

EXPECTED_SPLIT_CATEGORY = {
    ("development", "lookup"): 11,
    ("development", "multi-passage"): 6,
    ("development", "confusable"): 5,
    ("development", "unanswerable"): 8,
    ("held-out", "lookup"): 34,
    ("held-out", "multi-passage"): 19,
    ("held-out", "confusable"): 15,
    ("held-out", "unanswerable"): 22,
}

SCORING_RUBRIC = {
    "version": "UEL_RAG_SCORING_RUBRIC_V2",
    "unit_of_analysis": "one question and one system response",
    "factual_accuracy_0_1": {
        "0": "The answer contains a material factual error, contradiction, unsupported answer, or does not answer the question.",
        "1": "The answer is factually correct at the expected level and contains no material unsupported claim.",
    },
    "completeness_0_2": {
        "0": "Required points are absent or the response is materially wrong, contradictory, unsupported, or misleading.",
        "1": "At least one required point is correct, but one or more required points are missing without making the response materially misleading.",
        "2": "All required points are correctly covered at the expected level of detail.",
    },
    "claim_labels": {
        "fully_supported": "All material parts of the atomic factual claim are supported by the evidence being assessed.",
        "partially_supported": "The evidence supports part of the claim, but an added condition or detail is not fully justified.",
        "unsupported": "No assessed evidence supports the material claim.",
        "contradicted": "The assessed evidence states something materially inconsistent with the claim.",
        "ambiguous_not_evaluable": "The wording or evidence does not permit a reliable support judgement; exclude it from rate denominators.",
        "non_factual": "Politeness, transition, or another statement without a checkable proposition; exclude it from rate denominators.",
    },
    "claim_segmentation": "Split a response into the smallest independently checkable factual propositions. Keep linked conditions together when separating them would change their meaning.",
    "corpus_rates": {
        "corpus_support_rate": "fully corpus-supported claims / evaluable factual claims",
        "partial_support_rate": "partially corpus-supported claims / evaluable factual claims",
        "hallucination_rate": "(unsupported + contradicted claims) / evaluable factual claims",
    },
    "rag_context_rates": {
        "context_grounding_rate": "fully retrieved-context-supported claims / context-evaluable factual claims",
        "context_partial_support_rate": "partially retrieved-context-supported claims / context-evaluable factual claims",
        "context_hallucination_rate": "(unsupported + contradicted by context) / context-evaluable factual claims",
        "citation_correctness_rate": "citations that support the attached claim / citations assessed",
        "citation_coverage_rate": "evaluable factual claims with a citation / evaluable factual claims",
    },
    "refusal_rules": {
        "correct_refusal": "For an unanswerable item, clearly states that the supplied corpus is insufficient and adds no substantive guess.",
        "false_answer": "For an unanswerable item, asserts substantive information that is unsupported by the frozen corpus.",
    },
    "missing_output_rule": "An empty model response is incorrect. One rerun is allowed only for a confirmed infrastructure exception; a second failure remains missing.",
}

SCORING_EXAMPLES = [
    {
        "example": "DEV_M_001 completeness 2",
        "response": "The student must pass all taught modules and the dissertation for 180 credits, and normally achieve at least 80% attendance across all modules.",
        "decision": "completeness=2; both required points are present",
    },
    {
        "example": "DEV_M_001 completeness 1",
        "response": "The student must pass all taught modules and the dissertation for 180 credits.",
        "decision": "completeness=1; the attendance requirement is omitted",
    },
    {
        "example": "DEV_M_001 completeness 0 / contradicted claim",
        "response": "The student only needs 50% attendance and does not need to pass the dissertation.",
        "decision": "completeness=0; the response materially contradicts the verified requirements",
    },
    {
        "example": "DEV_M_001 partially supported claim",
        "response": "The student needs 80% attendance in every individual module.",
        "decision": "partially supported; 80% across all modules is supported, but the added per-module condition is not",
    },
    {
        "example": "DEV_U_001 correct refusal",
        "response": REFUSAL_TEXT,
        "decision": "correct_refusal=1; no factual guess is added",
    },
    {
        "example": "DEV_U_001 false answer",
        "response": "The Docklands Library is open from 10:00 to 16:00 on Sundays.",
        "decision": "false_answer=1; the frozen corpus does not support these hours",
    },
]

print(json.dumps(asdict(CFG), indent=2))


## 3. Discover and read the authoritative inputs

CSV encoding is detected explicitly because the reviewed files may have been exported by Excel as Windows-1252. Every derived CSV is written as UTF-8. The notebook resolves files by their required content and fails on ambiguity instead of silently taking the first match.


In [ ]:
QUESTION_REQUIRED = {
    "question_number", "question_id", "question_text", "reference_answer", "evidence_text",
    "source_locations", "secondary_tags", "required_points", "validation_notes", "evidence_ids",
    "answerable", "primary_category", "split", "primary_document_id",
}
EVIDENCE_REQUIRED = {
    "evidence_id", "question_id", "required_point", "document_id", "page_section",
    "evidence_text", "matching_terms", "review_status", "review_notes", "matching_rule",
    "minimum_overlap_ratio",
}
MANIFEST_REQUIRED = {
    "document_id", "title", "filename", "include", "source_url", "access_date",
}


def read_csv_flexible(path: Path) -> tuple[pd.DataFrame, str]:
    errors = []
    for encoding in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            frame = pd.read_csv(path, encoding=encoding, dtype=str, keep_default_na=False)
            return frame, encoding
        except UnicodeDecodeError as exc:
            errors.append(f"{encoding}: {exc}")
    raise UnicodeError(f"Cannot decode {path}: {' | '.join(errors)}")


def locate_csv_by_schema(root: Path, required: set[str], label: str) -> Path:
    matches = []
    for path in root.rglob("*.csv"):
        try:
            frame, _ = read_csv_flexible(path)
        except Exception:
            continue
        if required.issubset(frame.columns):
            matches.append(path)
    if len(matches) != 1:
        raise ValueError(f"Expected one {label} CSV, found {len(matches)}: {matches}")
    return matches[0]


QUESTIONS_INPUT = locate_csv_by_schema(STAGED_INPUT_ROOT, QUESTION_REQUIRED, "questions")
EVIDENCE_INPUT = locate_csv_by_schema(STAGED_INPUT_ROOT, EVIDENCE_REQUIRED, "evidence")
MANIFEST_INPUT = locate_csv_by_schema(STAGED_INPUT_ROOT, MANIFEST_REQUIRED, "corpus manifest")

questions_raw, questions_encoding = read_csv_flexible(QUESTIONS_INPUT)
evidence_raw, evidence_encoding = read_csv_flexible(EVIDENCE_INPUT)
manifest_raw, manifest_encoding = read_csv_flexible(MANIFEST_INPUT)

document_candidates = [p for p in STAGED_INPUT_ROOT.rglob("*") if p.suffix.lower() in {".pdf", ".docx"}]
if len(document_candidates) != 15:
    raise ValueError(f"Expected exactly 15 PDF/DOCX documents, found {len(document_candidates)}")
DOCUMENT_FOLDER = Path(os.path.commonpath([str(p.parent) for p in document_candidates]))

print("Questions:", QUESTIONS_INPUT.name, questions_raw.shape, questions_encoding)
print("Evidence:", EVIDENCE_INPUT.name, evidence_raw.shape, evidence_encoding)
print("Manifest:", MANIFEST_INPUT.name, manifest_raw.shape, manifest_encoding)
print("Documents:", len(document_candidates), "in", DOCUMENT_FOLDER)


## 4. Deterministic cleaning, page maps and extraction audit

PDFs are extracted page by page with PyMuPDF. Repeated margin lines and isolated page numbers are removed using a documented corpus-wide rule. DOCX paragraphs and tables are read in document order. Cleaning normalises Unicode and whitespace but does not rewrite policy meaning.

A short list of known extraction artefacts was verified against the frozen originals during dataset review (for example, `subsequentto` and a missing separator between two PDF paragraphs). Each repair must occur exactly once and is written to `source_text_repairs.csv`. The original binary documents are copied unchanged and retain their own SHA-256 hashes.


In [ ]:
SOURCE_TEXT_REPAIRS = {
    "UEL002": [
        ("We will cancel Your access card and You will lose Your access to University systems, services and facilities.",
         "9.4. We will cancel Your access card and You will lose Your access to University systems, services and facilities."),
        ("You will need to return Your student card, and any other property owned by Us.",
         "9.5. You will need to return Your student card, and any other property owned by Us."),
    ],
    "UEL003": [
        ("UEL Direct .", "UEL Direct."),
        ("is engaged withProvisional results", "is engaged with.\n\nProvisional results"),
    ],
    "UEL005": [
        ("or re-\nregister for the module", "or re-register for the module"),
        ("Academic Regulations .", "Academic Regulations."),
        ("Academic\nRegulations\n.", "Academic\nRegulations."),
    ],
    "UEL006": [
        ("students.union@uel.ac.uk ,", "students.union@uel.ac.uk,"),
        ("Appeals (December 2022)\n, produced", "Appeals (December 2022), produced"),
    ],
    "UEL008": [("subsequentto", "subsequent to"), ("interalia", "inter alia")],
    "UEL009": [("Appeals\n(December 2022)\n, produced", "Appeals\n(December 2022), produced")],
    "UEL010": [
        ("The nature and extent of feedback the student may expect will be communicated to the student for each assessment task at the time it is set.",
         "3.1.3 The nature and extent of feedback the student may expect will be communicated to the student for each assessment task at the time it is set."),
    ],
}


def clean_text_block(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def recurring_margin_lines(raw_pages: list[str], ratio: float = 0.35) -> set[str]:
    counts: Counter[str] = Counter()
    for raw_page in raw_pages:
        lines = [clean_text_block(line) for line in raw_page.splitlines() if clean_text_block(line)]
        for candidate in set(lines[:2] + lines[-2:]):
            if 2 <= len(candidate) <= 160:
                counts[candidate] += 1
    threshold = max(3, math.ceil(len(raw_pages) * ratio))
    return {line for line, count in counts.items() if count >= threshold}


def extract_pdf_pages(path: Path) -> tuple[list[dict[str, Any]], list[str]]:
    with fitz.open(path) as pdf:
        raw_pages = [page.get_text("text", sort=True) for page in pdf]
    repeated = recurring_margin_lines(raw_pages)
    pages = []
    for page_number, raw_text in enumerate(raw_pages, start=1):
        kept = []
        for line in raw_text.splitlines():
            normal = clean_text_block(line)
            if normal in repeated:
                continue
            if re.fullmatch(r"(?:page\s*)?\d+(?:\s*of\s*\d+)?", normal, flags=re.I):
                continue
            kept.append(line)
        pages.append({"page": page_number, "section": "", "text": clean_text_block("\n".join(kept))})
    return pages, sorted(repeated)


def iter_docx_blocks(document: DocumentType) -> Iterable[Paragraph | Table]:
    for child in document.element.body.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, document)
        elif isinstance(child, CT_Tbl):
            yield Table(child, document)


def extract_docx_blocks(path: Path) -> tuple[list[dict[str, Any]], list[str]]:
    document = Document(path)
    parts = []
    current_section = ""
    for block in iter_docx_blocks(document):
        if isinstance(block, Paragraph):
            text = clean_text_block(block.text)
            if not text:
                continue
            if block.style and block.style.name.lower().startswith("heading"):
                current_section = text
            parts.append({"page": None, "section": current_section, "text": text})
        else:
            for row in block.rows:
                cells = [clean_text_block(cell.text) for cell in row.cells]
                if any(cells):
                    parts.append({"page": None, "section": current_section, "text": " | ".join(cells)})
    return parts, []


def apply_verified_repairs(document_id: str, parts: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    parts = [dict(item) for item in parts]
    audit = []
    for old, new in SOURCE_TEXT_REPAIRS.get(document_id, []):
        occurrences = sum(item["text"].count(old) for item in parts)
        if occurrences != 1:
            raise AssertionError(
                f"{document_id}: expected exactly one occurrence of verified extraction artefact {old!r}; found {occurrences}."
            )
        for item in parts:
            item["text"] = item["text"].replace(old, new)
        audit.append({
            "document_id": document_id,
            "original_extraction": old,
            "cleaned_replacement": new,
            "occurrences": occurrences,
            "reason": "Source-verified extraction repair; policy meaning unchanged",
        })
    return parts, audit


def assemble_clean_document(parts: list[dict[str, Any]]) -> tuple[str, list[dict[str, Any]]]:
    output, page_map, cursor = [], [], 0
    for part in parts:
        text = part["text"].strip()
        if not text:
            continue
        if output:
            output.append("\n\n")
            cursor += 2
        start = cursor
        output.append(text)
        cursor += len(text)
        page_map.append({
            "page": part.get("page"),
            "section": part.get("section", ""),
            "clean_start": start,
            "clean_end": cursor,
        })
    return "".join(output), page_map


In [ ]:
def build_clean_corpus(manifest: pd.DataFrame, document_folder: Path, build_dir: Path):
    clean_dir = build_dir / "cleaned_text"
    map_dir = build_dir / "page_maps"
    source_dir = build_dir / "source_documents"
    for path in (clean_dir, map_dir, source_dir):
        path.mkdir(parents=True, exist_ok=True)

    manifest_rows, document_rows, map_rows = [], [], []
    extraction_rows, repair_rows = [], []
    for row in manifest.sort_values("document_id").to_dict("records"):
        document_id = row["document_id"].strip()
        source_path = document_folder / row["filename"]
        if not source_path.exists():
            candidates = [p for p in document_folder.rglob(row["filename"])]
            if len(candidates) != 1:
                raise FileNotFoundError(f"Cannot resolve manifest file {row['filename']}")
            source_path = candidates[0]

        if source_path.suffix.lower() == ".pdf":
            parts, removed_margins = extract_pdf_pages(source_path)
        elif source_path.suffix.lower() == ".docx":
            parts, removed_margins = extract_docx_blocks(source_path)
        else:
            raise ValueError(f"Unsupported source type: {source_path.name}")

        parts, repairs = apply_verified_repairs(document_id, parts)
        clean_text, page_map = assemble_clean_document(parts)
        if not clean_text:
            raise ValueError(f"No text extracted from {source_path.name}")

        clean_path = clean_dir / f"{document_id}.txt"
        clean_path.write_text(clean_text, encoding="utf-8")
        copied_source = source_dir / source_path.name
        shutil.copy2(source_path, copied_source)

        for item in page_map:
            item.update({"document_id": document_id})
            map_rows.append(item)
        (map_dir / f"{document_id}_page_map.json").write_text(
            json.dumps(page_map, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        for repair in repairs:
            repair_rows.append(repair)

        enriched = dict(row)
        enriched.update({
            "sha256": sha256_file(source_path),
            "file_size_bytes": source_path.stat().st_size,
            "cleaned_characters": len(clean_text),
            "cleaned_words": len(re.findall(r"\b\w+\b", clean_text, flags=re.UNICODE)),
            "page_map_records": len(page_map),
            "file_type": source_path.suffix.lower().lstrip("."),
        })
        manifest_rows.append(enriched)
        document_rows.append({
            "document_id": document_id,
            "filename": source_path.name,
            "source_sha256": enriched["sha256"],
            "clean_text_path": f"cleaned_text/{document_id}.txt",
            "clean_text_sha256": sha256_file(clean_path),
            "cleaned_characters": len(clean_text),
            "clean_text": clean_text,
        })
        extraction_rows.append({
            "document_id": document_id,
            "filename": source_path.name,
            "file_type": enriched["file_type"],
            "extraction_status": "PROGRAMMATIC_PASS",
            "raw_units": len(parts),
            "page_map_records": len(page_map),
            "removed_margin_lines": json.dumps(removed_margins, ensure_ascii=False),
            "verified_repairs": len(repairs),
            "cleaned_characters": len(clean_text),
        })

    manifest_enriched = pd.DataFrame(manifest_rows)
    documents = pd.DataFrame(document_rows)
    page_maps = pd.DataFrame(map_rows)[
        ["document_id", "page", "section", "clean_start", "clean_end"]
    ]
    extraction_audit = pd.DataFrame(extraction_rows)
    repair_audit = pd.DataFrame(repair_rows, columns=[
        "document_id", "original_extraction", "cleaned_replacement", "occurrences", "reason"
    ])
    return manifest_enriched, documents, page_maps, extraction_audit, repair_audit


## 5. Generate exact evidence offsets

Each reviewed evidence passage is aligned to its named cleaned document. Alignment first uses a punctuation-aware canonical character representation, then a word-token sequence fallback that tolerates layout punctuation without changing words. A passage must have exactly one match. `clean_start` and `clean_end` point to the exact substring saved in `matched_clean_text`.


In [ ]:
def canonical_characters_with_map(text: str) -> tuple[str, list[int]]:
    characters, source_positions = [], []
    previous_space = True
    quote_map = {"“": '"', "”": '"', "‘": "'", "’": "'"}
    dash_chars = {"–", "—", "−"}
    for index, character in enumerate(unicodedata.normalize("NFKC", text)):
        character = quote_map.get(character, character)
        if character in dash_chars:
            character = "-"
        if character.isspace():
            if not previous_space:
                characters.append(" ")
                source_positions.append(index)
            previous_space = True
        else:
            characters.append(character.casefold())
            source_positions.append(index)
            previous_space = False
    while characters and characters[-1] == " ":
        characters.pop()
        source_positions.pop()
    return "".join(characters), source_positions


def normal_token(token: str) -> str:
    return unicodedata.normalize("NFKC", token).casefold().replace("’", "'")


def token_spans(text: str) -> list[tuple[str, int, int]]:
    pattern = r"[^\W_]+(?:[’'][^\W_]+)*"
    return [(normal_token(match.group(0)), match.start(), match.end())
            for match in re.finditer(pattern, text, flags=re.UNICODE)]


def exact_canonical_matches(source: str, target: str) -> list[tuple[int, int]]:
    canonical_source, positions = canonical_characters_with_map(source)
    canonical_target, _ = canonical_characters_with_map(target)
    if not canonical_target:
        return []
    matches, cursor = [], 0
    while True:
        found = canonical_source.find(canonical_target, cursor)
        if found < 0:
            break
        start = positions[found]
        end = positions[found + len(canonical_target) - 1] + 1
        matches.append((start, end))
        cursor = found + 1
    return matches


def token_sequence_matches(source: str, target: str) -> list[tuple[int, int]]:
    source_tokens = token_spans(source)
    target_tokens = [token for token, _, _ in token_spans(target)]
    if not target_tokens:
        return []
    first, width = target_tokens[0], len(target_tokens)
    output = []
    for index, (token, start, _) in enumerate(source_tokens):
        if token != first or index + width > len(source_tokens):
            continue
        candidate = [item[0] for item in source_tokens[index:index + width]]
        if candidate == target_tokens:
            output.append((start, source_tokens[index + width - 1][2]))
    return output


def align_one_passage(source: str, target: str) -> tuple[int, int, str]:
    matches = exact_canonical_matches(source, target)
    method = "canonical_character"
    if not matches:
        matches = token_sequence_matches(source, target)
        method = "word_token_sequence"
    if len(matches) != 1:
        raise ValueError(f"Evidence passage must align uniquely; found {len(matches)} matches")
    start, end = matches[0]
    return start, end, method


def page_for_offset(page_maps: pd.DataFrame, document_id: str, offset: int) -> tuple[Any, str]:
    subset = page_maps[
        (page_maps["document_id"] == document_id)
        & (page_maps["clean_start"] <= offset)
        & (page_maps["clean_end"] > offset)
    ]
    if subset.empty:
        return "", ""
    row = subset.iloc[0]
    return row["page"], row["section"]


def align_evidence_units(evidence: pd.DataFrame, documents: pd.DataFrame, page_maps: pd.DataFrame) -> pd.DataFrame:
    clean_lookup = documents.set_index("document_id")["clean_text"].to_dict()
    rows = []
    for row in tqdm(evidence.to_dict("records"), desc="Aligning evidence"):
        document_id = row["document_id"].strip()
        if document_id not in clean_lookup:
            raise KeyError(f"Unknown evidence document: {document_id}")
        start, end, method = align_one_passage(clean_lookup[document_id], row["evidence_text"])
        page, section = page_for_offset(page_maps, document_id, start)
        enriched = dict(row)
        enriched.update({
            "clean_start": start,
            "clean_end": end,
            "matched_clean_text": clean_lookup[document_id][start:end],
            "alignment_method": method,
            "alignment_occurrences": 1,
            "mapped_page": page,
            "mapped_section": section,
        })
        rows.append(enriched)
    return pd.DataFrame(rows)


## 6. Validate the complete dataset

Validation covers schema, counts, split/category distribution, JSON-list fields, answerability rules, evidence relationships, exact offset slices, matching terms, multi-passage separation, absence of reviewer metadata and the manually confirmed unanswerable-question records. A failed check stops freezing.


In [ ]:
JSON_LIST_COLUMNS = ["source_locations", "secondary_tags", "required_points", "evidence_ids"]


def parse_json_list(value: Any, label: str) -> list[Any]:
    if isinstance(value, list):
        return value
    if value is None or str(value).strip() == "":
        return []
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON list in {label}: {exc}") from exc
    if not isinstance(parsed, list):
        raise ValueError(f"{label} must contain a JSON list")
    return parsed


def parse_bool(value: Any) -> bool:
    normalized = str(value).strip().casefold()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot parse boolean value: {value!r}")


def split_question_evidence(value: Any) -> list[str]:
    text = str(value).strip()
    if not text:
        return []
    return [passage.strip() for passage in re.split(r"\n\s*---\s*\n", text) if passage.strip()]


def canonical_words(text: Any) -> list[str]:
    return [item[0] for item in token_spans(str(text))]


def validate_dataset(
    questions: pd.DataFrame,
    evidence: pd.DataFrame,
    manifest: pd.DataFrame,
    documents: pd.DataFrame,
    page_maps: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    checks: list[dict[str, Any]] = []

    def record(name: str, passed: bool, detail: str) -> None:
        checks.append({"check": name, "status": "PASS" if passed else "FAIL", "detail": detail})

    record("document_count", len(manifest) == 15 == len(documents), f"manifest={len(manifest)}, extracted={len(documents)}")
    record("question_count", len(questions) == 120, f"rows={len(questions)}")
    record("evidence_count", len(evidence) == 143, f"rows={len(evidence)}")
    record("unique_question_ids", questions["question_id"].nunique() == 120, f"unique={questions['question_id'].nunique()}")
    record("unique_evidence_ids", evidence["evidence_id"].nunique() == 143, f"unique={evidence['evidence_id'].nunique()}")
    record("unique_document_ids", manifest["document_id"].nunique() == 15, f"unique={manifest['document_id'].nunique()}")

    reviewer_columns = [column for column in questions.columns if "reviewer" in column.casefold()]
    reviewer_text = questions.astype(str).apply(lambda col: col.str.contains("Reviewer A|Reviewer B", case=False, regex=True)).any().any()
    record("no_independent_reviewer_fields", not reviewer_columns and not reviewer_text,
           f"columns={reviewer_columns}, text_reference={reviewer_text}")

    split_category = questions.groupby(["split", "primary_category"]).size().to_dict()
    record("expected_split_category_counts", split_category == EXPECTED_SPLIT_CATEGORY, str(split_category))
    record("development_heldout_counts",
           questions["split"].value_counts().to_dict() == {"held-out": 90, "development": 30},
           str(questions["split"].value_counts().to_dict()))

    json_errors, relationship_errors = [], []
    evidence_id_set = set(evidence["evidence_id"])
    evidence_by_id = evidence.set_index("evidence_id").to_dict("index")
    question_answerable = {}
    for row in questions.to_dict("records"):
        qid = row["question_id"]
        try:
            parsed = {column: parse_json_list(row[column], f"{qid}.{column}") for column in JSON_LIST_COLUMNS}
        except ValueError as exc:
            json_errors.append(str(exc))
            continue
        try:
            answerable = parse_bool(row["answerable"])
        except ValueError as exc:
            json_errors.append(f"{qid}: {exc}")
            continue
        question_answerable[qid] = answerable
        evidence_passages = split_question_evidence(row["evidence_text"])
        lengths = [len(parsed["evidence_ids"]), len(evidence_passages), len(parsed["source_locations"]), len(parsed["required_points"])]
        if answerable and len(set(lengths)) != 1:
            relationship_errors.append(f"{qid}: evidence/location/point counts differ: {lengths}")
        if answerable and not parsed["evidence_ids"]:
            relationship_errors.append(f"{qid}: answerable question has no evidence")
        if not answerable and any(lengths):
            relationship_errors.append(f"{qid}: unanswerable question has populated evidence fields")
        missing_ids = set(parsed["evidence_ids"]) - evidence_id_set
        if missing_ids:
            relationship_errors.append(f"{qid}: unknown evidence IDs {sorted(missing_ids)}")
        for evidence_id in parsed["evidence_ids"]:
            if evidence_by_id[evidence_id]["question_id"] != qid:
                relationship_errors.append(f"{qid}: {evidence_id} points to another question")
        if row["primary_category"] == "multi-passage":
            spans = [(evidence_by_id[item]["document_id"], int(evidence_by_id[item]["clean_start"]), int(evidence_by_id[item]["clean_end"]))
                     for item in parsed["evidence_ids"]]
            if len(spans) < 2 or len(set(spans)) != len(spans):
                relationship_errors.append(f"{qid}: multi-passage evidence spans are not distinct")

    record("json_list_fields", not json_errors, "none" if not json_errors else " | ".join(json_errors[:10]))
    record("question_evidence_relationships", not relationship_errors,
           "none" if not relationship_errors else " | ".join(relationship_errors[:10]))

    offset_errors, term_errors, rule_errors = [], [], []
    clean_lookup = documents.set_index("document_id")["clean_text"].to_dict()
    valid_question_ids = set(questions["question_id"])
    for row in evidence.to_dict("records"):
        eid, document_id = row["evidence_id"], row["document_id"]
        start, end = int(row["clean_start"]), int(row["clean_end"])
        source = clean_lookup.get(document_id, "")
        exact_slice = source[start:end]
        if not (0 <= start < end <= len(source)) or exact_slice != row["matched_clean_text"]:
            offset_errors.append(eid)
        if row["question_id"] not in valid_question_ids:
            relationship_errors.append(f"{eid}: unknown question ID")
        try:
            terms = parse_json_list(row["matching_terms"], f"{eid}.matching_terms")
        except ValueError as exc:
            term_errors.append(str(exc))
            continue
        if str(row.get("matching_rule", "")).strip() != "offset_overlap_and_terms":
            rule_errors.append(f"{eid}: matching_rule must be offset_overlap_and_terms")
        try:
            threshold = float(row.get("minimum_overlap_ratio", CFG.evidence_overlap_threshold))
            if not 0 < threshold <= 1:
                raise ValueError
        except (TypeError, ValueError):
            rule_errors.append(f"{eid}: minimum_overlap_ratio must be in (0, 1]")
        if not terms:
            rule_errors.append(f"{eid}: matching_terms must not be empty")
        slice_words = canonical_words(exact_slice)
        joined = " ".join(slice_words)
        missing = [term for term in terms if " ".join(canonical_words(term)) not in joined]
        if missing:
            term_errors.append(f"{eid}: {missing}")

    record("exact_offset_slices", not offset_errors, "none" if not offset_errors else str(offset_errors[:20]))
    record("matching_terms_present", not term_errors, "none" if not term_errors else " | ".join(term_errors[:10]))
    record("evidence_matching_rules", not rule_errors,
           "none" if not rule_errors else " | ".join(rule_errors[:10]))

    unanswerable = questions[questions["primary_category"] == "unanswerable"]
    answerability_flags = questions["answerable"].map(parse_bool)
    category_flags = questions["primary_category"].eq("unanswerable")
    answerability_ok = (answerability_flags == ~category_flags).all()
    record("category_answerability_consistency", bool(answerability_ok),
           "unanswerable rows must be False; all other categories must be True")
    notes_ok = unanswerable["validation_notes"].str.contains(
        "manually reviewed against all 15 frozen source documents", case=False, regex=False
    ).all()
    record("unanswerable_researcher_validation", len(unanswerable) == 30 and notes_ok,
           f"rows={len(unanswerable)}, all_notes_confirmed={notes_ok}")

    corpus_word_strings = {
        document_id: " ".join(canonical_words(text)) for document_id, text in clean_lookup.items()
    }
    supported_negative_queries = []
    for row in unanswerable.to_dict("records"):
        tags = parse_json_list(row["secondary_tags"], f"{row['question_id']}.secondary_tags")
        negative_tags = [str(tag).split(":", 1)[1] for tag in tags if str(tag).casefold().startswith("negative-query:")]
        if len(negative_tags) != 1:
            supported_negative_queries.append(f"{row['question_id']}: expected one negative-query tag")
            continue
        phrase = " ".join(canonical_words(negative_tags[0]))
        matching_documents = [doc_id for doc_id, corpus_words in corpus_word_strings.items() if phrase and phrase in corpus_words]
        if matching_documents:
            supported_negative_queries.append(f"{row['question_id']}: exact phrase found in {matching_documents}")
    record("unanswerable_exact_phrase_absence", not supported_negative_queries,
           "none" if not supported_negative_queries else " | ".join(supported_negative_queries[:10]))

    checks_frame = pd.DataFrame(checks)
    report = {
        "validation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "status": "PASS" if checks_frame["status"].eq("PASS").all() else "FAIL",
        "checks": checks,
        "counts": {
            "documents": len(documents), "questions": len(questions), "evidence_units": len(evidence),
            "development": int((questions["split"] == "development").sum()),
            "held_out": int((questions["split"] == "held-out").sum()),
        },
    }
    if report["status"] != "PASS":
        failed = checks_frame[checks_frame["status"] == "FAIL"]
        raise ValueError("Dataset validation failed:\n" + failed.to_string(index=False))
    return checks_frame, report


## 7. Build and freeze the derived corpus snapshot

This cell creates the enriched CSVs and clean artefacts, validates them, and copies the complete result to `02_frozen_dataset`. The freeze manifest records the SHA-256 of every file. Later cells read only this frozen folder. Re-running against an existing freeze verifies it but never overwrites it.


In [ ]:
def write_utf8_csv(frame: pd.DataFrame, path: Path) -> None:
    frame.to_csv(path, index=False, encoding="utf-8", lineterminator="\n")


def serialise_question_lists(questions: pd.DataFrame) -> pd.DataFrame:
    output = questions.copy()
    for column in JSON_LIST_COLUMNS:
        output[column] = output[column].map(
            lambda value: json.dumps(parse_json_list(value, column), ensure_ascii=False)
        )
    output["answerable"] = output["answerable"].map(parse_bool)
    return output


def file_inventory(root: Path, excluded_names: set[str] | None = None) -> list[dict[str, Any]]:
    excluded_names = excluded_names or set()
    records = []
    for path in sorted(item for item in root.rglob("*") if item.is_file() and item.name not in excluded_names):
        records.append({
            "relative_path": path.relative_to(root).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
    return records


def verify_frozen_folder(frozen_dir: Path) -> dict[str, Any]:
    manifest_path = frozen_dir / "freeze_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Incomplete freeze: {manifest_path} is missing")
    freeze = json.loads(manifest_path.read_text(encoding="utf-8"))
    errors = []
    for item in freeze["files"]:
        path = frozen_dir / item["relative_path"]
        if not path.exists():
            errors.append(f"missing {item['relative_path']}")
        elif path.stat().st_size != item["size_bytes"] or sha256_file(path) != item["sha256"]:
            errors.append(f"hash/size mismatch {item['relative_path']}")
    if errors:
        raise ValueError("Frozen dataset verification failed: " + "; ".join(errors))
    print(f"Frozen dataset verified: {len(freeze['files'])} files")
    return freeze


def build_and_freeze_dataset():
    frozen_manifest_path = DIRS["frozen"] / "freeze_manifest.json"
    if frozen_manifest_path.exists():
        return verify_frozen_folder(DIRS["frozen"])
    if any(DIRS["frozen"].iterdir()):
        raise FileExistsError("Frozen dataset folder is non-empty but has no valid freeze manifest.")

    manifest = manifest_raw.copy()
    manifest["include"] = manifest["include"].map(parse_bool)
    if not manifest["include"].all():
        raise ValueError("All 15 supplied corpus rows must be included for this frozen evaluation.")

    questions = serialise_question_lists(questions_raw)
    evidence = evidence_raw.copy()
    manifest_enriched, documents, page_maps, extraction_audit, repair_audit = build_clean_corpus(
        manifest, DOCUMENT_FOLDER, DIRS["build"]
    )
    evidence_enriched = align_evidence_units(evidence, documents, page_maps)
    checks, validation_report = validate_dataset(
        questions, evidence_enriched, manifest_enriched, documents, page_maps
    )

    write_utf8_csv(questions, DIRS["build"] / "questions.csv")
    write_utf8_csv(evidence_enriched, DIRS["build"] / "evidence_units_enriched.csv")
    write_utf8_csv(manifest_enriched, DIRS["build"] / "corpus_manifest_enriched.csv")
    write_utf8_csv(documents.drop(columns=["clean_text"]), DIRS["build"] / "document_register.csv")
    write_utf8_csv(page_maps, DIRS["build"] / "page_map.csv")
    write_utf8_csv(extraction_audit, DIRS["build"] / "extraction_audit.csv")
    write_utf8_csv(repair_audit, DIRS["build"] / "source_text_repairs.csv")
    write_utf8_csv(checks, DIRS["build"] / "validation_checks.csv")
    (DIRS["build"] / "validation_report.json").write_text(
        json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    input_hashes = {
        "questions_input_sha256": sha256_file(QUESTIONS_INPUT),
        "evidence_input_sha256": sha256_file(EVIDENCE_INPUT),
        "manifest_input_sha256": sha256_file(MANIFEST_INPUT),
    }
    shutil.copytree(DIRS["build"], DIRS["frozen"], dirs_exist_ok=True)
    freeze = {
        "freeze_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "corpus_version": CFG.corpus_version,
        "question_dataset_version": CFG.question_dataset_version,
        "input_hashes": input_hashes,
        "validation_status": validation_report["status"],
        "files": file_inventory(DIRS["frozen"], {"freeze_manifest.json", "FREEZE_COMPLETE.txt"}),
        "rule": "This folder is a derived immutable snapshot. Do not edit files in place; create a new version and freeze manifest.",
    }
    canonical = json.dumps(freeze, sort_keys=True, ensure_ascii=False)
    freeze["freeze_record_sha256"] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    (DIRS["frozen"] / "freeze_manifest.json").write_text(
        json.dumps(freeze, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (DIRS["frozen"] / "FREEZE_COMPLETE.txt").write_text(
        f"Frozen {freeze['freeze_timestamp_utc']}\nRecord SHA-256: {freeze['freeze_record_sha256']}\n",
        encoding="utf-8",
    )
    return verify_frozen_folder(DIRS["frozen"])


DATASET_FREEZE = build_and_freeze_dataset()
print("Validation: PASS")
print("Frozen record SHA-256:", DATASET_FREEZE["freeze_record_sha256"])


In [ ]:
def read_frozen_dataset():
    verify_frozen_folder(DIRS["frozen"])
    questions = pd.read_csv(DIRS["frozen"] / "questions.csv", dtype=str, keep_default_na=False)
    questions["answerable"] = questions["answerable"].map(parse_bool)
    evidence = pd.read_csv(DIRS["frozen"] / "evidence_units_enriched.csv", dtype=str, keep_default_na=False)
    evidence["clean_start"] = evidence["clean_start"].astype(int)
    evidence["clean_end"] = evidence["clean_end"].astype(int)
    evidence["minimum_overlap_ratio"] = pd.to_numeric(evidence["minimum_overlap_ratio"], errors="coerce").fillna(CFG.evidence_overlap_threshold)
    manifest = pd.read_csv(DIRS["frozen"] / "corpus_manifest_enriched.csv", dtype=str, keep_default_na=False)
    page_maps = pd.read_csv(DIRS["frozen"] / "page_map.csv", dtype=str, keep_default_na=False)
    page_maps["clean_start"] = page_maps["clean_start"].astype(int)
    page_maps["clean_end"] = page_maps["clean_end"].astype(int)
    documents = pd.read_csv(DIRS["frozen"] / "document_register.csv", dtype=str, keep_default_na=False)
    documents["clean_text"] = documents["document_id"].map(
        lambda document_id: (DIRS["frozen"] / "cleaned_text" / f"{document_id}.txt").read_text(encoding="utf-8")
    )
    return questions, evidence, manifest, documents, page_maps


questions_df, evidence_df, corpus_manifest_df, documents_df, page_map_df = read_frozen_dataset()
display(corpus_manifest_df[["document_id", "filename", "sha256", "file_size_bytes", "cleaned_characters"]])
display(pd.crosstab(questions_df["primary_category"], questions_df["split"], margins=True))


## 8. Controlled retrieval implementation

Chunks are defined with the fixed BGE tokenizer and retain clean-text character provenance. Embeddings are L2-normalised and searched with FAISS inner product (cosine-equivalent under normalisation). Reranking uses the fixed cross-encoder. Index construction and model-loading time are excluded from per-query retrieval latency; query encoding, search and reranking are included.


In [ ]:
PINNED_MODEL_REVISIONS = {
    "generator": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
    "embedding": "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    "reranker": "233902d25c440f23af6f7d6e94d2946bac0bee0a"
}

def resolve_model_revisions() -> dict[str, str]:
    """Return the immutable Hugging Face revisions used in the completed frozen run."""
    return dict(PINNED_MODEL_REVISIONS)


MODEL_REVISIONS = resolve_model_revisions()
print("Resolved immutable model revisions:")
print(json.dumps(MODEL_REVISIONS, indent=2))

embedding_tokenizer = None
embedding_model = None
reranker_model = None
CHUNK_CACHE: dict[tuple[int, int], pd.DataFrame] = {}
INDEX_CACHE: dict[tuple[int, int], tuple[pd.DataFrame, Any]] = {}


def load_retrieval_models(load_reranker: bool = True):
    global embedding_tokenizer, embedding_model, reranker_model
    if embedding_tokenizer is None:
        embedding_tokenizer = AutoTokenizer.from_pretrained(
            CFG.embedding_model, revision=MODEL_REVISIONS["embedding"], use_fast=True
        )
    if embedding_model is None:
        embedding_model = SentenceTransformer(
            CFG.embedding_model, revision=MODEL_REVISIONS["embedding"]
        )
        embedding_model.eval()
    if load_reranker and reranker_model is None:
        reranker_model = CrossEncoder(
            CFG.reranker_model, revision=MODEL_REVISIONS["reranker"]
        )
    return embedding_tokenizer, embedding_model, reranker_model


def page_location(document_id: str, char_start: int, maps: pd.DataFrame) -> tuple[str, str]:
    rows = maps[
        (maps["document_id"] == document_id)
        & (maps["clean_start"] <= char_start)
        & (maps["clean_end"] > char_start)
    ]
    if rows.empty:
        return "", ""
    row = rows.iloc[0]
    return str(row["page"]), str(row["section"])


def chunk_documents(documents: pd.DataFrame, maps: pd.DataFrame, chunk_size: int, overlap_pct: int) -> pd.DataFrame:
    key = (chunk_size, overlap_pct)
    if key in CHUNK_CACHE:
        return CHUNK_CACHE[key]
    tokenizer, _, _ = load_retrieval_models(load_reranker=False)
    overlap = int(round(chunk_size * overlap_pct / 100))
    stride = chunk_size - overlap
    if stride <= 0:
        raise ValueError("Chunk stride must be positive")
    rows = []
    for document in documents.sort_values("document_id").to_dict("records"):
        encoded = tokenizer(
            document["clean_text"], add_special_tokens=False, return_offsets_mapping=True,
            truncation=False,
        )
        offsets = encoded["offset_mapping"]
        chunk_number = 0
        for token_start in range(0, len(offsets), stride):
            token_end = min(token_start + chunk_size, len(offsets))
            if token_start >= token_end:
                break
            char_start = int(offsets[token_start][0])
            char_end = int(offsets[token_end - 1][1])
            text = document["clean_text"][char_start:char_end].strip()
            if not text:
                continue
            page, section = page_location(document["document_id"], char_start, maps)
            rows.append({
                "chunk_id": f"{document['document_id']}::c{chunk_number:04d}",
                "document_id": document["document_id"],
                "chunk_number": chunk_number,
                "token_start": token_start,
                "token_end": token_end,
                "clean_start": char_start,
                "clean_end": char_end,
                "page": page,
                "section": section,
                "text": text,
            })
            chunk_number += 1
            if token_end == len(offsets):
                break
    chunks = pd.DataFrame(rows)
    CHUNK_CACHE[key] = chunks
    return chunks


def build_index(documents: pd.DataFrame, maps: pd.DataFrame, chunk_size: int, overlap_pct: int):
    key = (chunk_size, overlap_pct)
    if key in INDEX_CACHE:
        return INDEX_CACHE[key]
    chunks = chunk_documents(documents, maps, chunk_size, overlap_pct).reset_index(drop=True)
    _, model, _ = load_retrieval_models(load_reranker=False)
    embeddings = model.encode(
        chunks["text"].tolist(), batch_size=CFG.embedding_batch_size,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    INDEX_CACHE[key] = (chunks, index)
    return chunks, index


def retrieve(
    query: str,
    documents: pd.DataFrame,
    maps: pd.DataFrame,
    chunk_size: int,
    overlap_pct: int,
    rerank: bool,
    candidate_pool: int,
    top_k: int = 5,
) -> list[dict[str, Any]]:
    chunks, index = build_index(documents, maps, chunk_size, overlap_pct)
    _, model, cross_encoder = load_retrieval_models(load_reranker=rerank)
    search_width = max(top_k, candidate_pool if rerank else top_k)
    query_embedding = model.encode(
        [CFG.query_instruction + query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, indices = index.search(query_embedding, search_width)
    candidates = chunks.iloc[indices[0]].copy()
    candidates["embedding_score"] = scores[0]
    if rerank:
        pairs = [[query, text] for text in candidates["text"]]
        candidates["reranker_score"] = cross_encoder.predict(pairs, show_progress_bar=False)
        candidates = candidates.sort_values("reranker_score", ascending=False, kind="stable")
    else:
        candidates["reranker_score"] = np.nan
    candidates = candidates.head(top_k).reset_index(drop=True)
    candidates["rank"] = np.arange(1, len(candidates) + 1)
    return candidates.to_dict("records")


In [ ]:
def interval_overlap_ratio(chunk_start: int, chunk_end: int, evidence_start: int, evidence_end: int) -> float:
    overlap = max(0, min(chunk_end, evidence_end) - max(chunk_start, evidence_start))
    evidence_width = max(1, evidence_end - evidence_start)
    return overlap / evidence_width


def chunk_matches_evidence(chunk: dict[str, Any], evidence: dict[str, Any]) -> bool:
    if chunk["document_id"] != evidence["document_id"]:
        return False
    rule = str(evidence.get("matching_rule", "")).strip()
    if rule != "offset_overlap_and_terms":
        raise ValueError(f"Unsupported matching rule for {evidence.get('evidence_id')}: {rule}")
    ratio = interval_overlap_ratio(
        int(chunk["clean_start"]), int(chunk["clean_end"]),
        int(evidence["clean_start"]), int(evidence["clean_end"]),
    )
    threshold = float(evidence.get("minimum_overlap_ratio", CFG.evidence_overlap_threshold))
    if ratio < threshold:
        return False
    terms = parse_json_list(evidence.get("matching_terms", "[]"), f"{evidence.get('evidence_id')}.matching_terms")
    chunk_words = " ".join(canonical_words(chunk.get("text", "")))
    return all(" ".join(canonical_words(term)) in chunk_words for term in terms)


def retrieval_metrics_for_question(retrieved: list[dict[str, Any]], question_id: str, evidence: pd.DataFrame) -> dict[str, Any]:
    gold = evidence[evidence["question_id"] == question_id].to_dict("records")
    if not gold:
        return {
            "hit_rate_at_5": np.nan, "recall_at_5": np.nan, "mrr": np.nan,
            "matched_evidence_count": 0, "gold_evidence_count": 0,
            "matched_evidence_ids": [],
        }
    matched_ids, first_relevant_rank = [], None
    for gold_item in gold:
        supporting_ranks = [int(chunk["rank"]) for chunk in retrieved if chunk_matches_evidence(chunk, gold_item)]
        if supporting_ranks:
            matched_ids.append(gold_item["evidence_id"])
            best_rank = min(supporting_ranks)
            first_relevant_rank = best_rank if first_relevant_rank is None else min(first_relevant_rank, best_rank)
    return {
        "hit_rate_at_5": float(bool(matched_ids)),
        "recall_at_5": len(matched_ids) / len(gold),
        "mrr": 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank,
        "matched_evidence_count": len(matched_ids),
        "gold_evidence_count": len(gold),
        "matched_evidence_ids": matched_ids,
    }


def run_retrieval_config(config: dict[str, Any], questions: pd.DataFrame) -> pd.DataFrame:
    checkpoint = DIRS["development"] / f"{config['config_id']}_retrieval_checkpoint.csv"
    expected_ids = set(questions["question_id"])
    expected_config_json = json.dumps(config, sort_keys=True)
    if checkpoint.exists():
        saved = pd.read_csv(checkpoint, dtype=str, keep_default_na=False)
        if set(saved["question_id"]) != expected_ids or saved["question_id"].duplicated().any():
            raise ValueError(f"Invalid development checkpoint: {checkpoint}")
        if set(saved.get("config_json", [])) != {expected_config_json}:
            raise ValueError(f"Development checkpoint configuration mismatch: {checkpoint}")
        numeric_columns = [
            "latency_seconds", "hit_rate_at_5", "recall_at_5", "mrr",
            "matched_evidence_count", "gold_evidence_count",
        ]
        for column in numeric_columns:
            saved[column] = pd.to_numeric(saved[column], errors="coerce")
        print("Reusing verified development checkpoint:", checkpoint.name)
        return saved
    build_index(documents_df, page_map_df, config["chunk_size"], config["overlap_pct"])
    if config["rerank"]:
        load_retrieval_models(load_reranker=True)
    rows = []
    for row in tqdm(questions.sort_values("question_id").to_dict("records"), desc=config["config_id"]):
        started = time.perf_counter()
        retrieved = retrieve(
            row["question_text"], documents_df, page_map_df,
            chunk_size=config["chunk_size"], overlap_pct=config["overlap_pct"],
            rerank=config["rerank"], candidate_pool=config["candidate_pool"], top_k=CFG.top_k,
        )
        elapsed = time.perf_counter() - started
        metrics = retrieval_metrics_for_question(retrieved, row["question_id"], evidence_df)
        rows.append({
            "config_id": config["config_id"], "question_id": row["question_id"],
            "config_json": expected_config_json,
            "primary_category": row["primary_category"], "latency_seconds": elapsed,
            "retrieved_json": json.dumps(retrieved, ensure_ascii=False, default=str), **metrics,
        })
    result = pd.DataFrame(rows)
    write_utf8_csv(result, checkpoint)
    return result


def summarise_retrieval(rows: pd.DataFrame) -> pd.DataFrame:
    return (
        rows.groupby("config_id", as_index=False)
        .agg(
            mean_recall_at_5=("recall_at_5", "mean"),
            mean_mrr=("mrr", "mean"),
            mean_hit_rate_at_5=("hit_rate_at_5", "mean"),
            median_latency_seconds=("latency_seconds", "median"),
            questions=("question_id", "nunique"),
        )
    )


BASE_CONFIG = {"config_id": "D0", "chunk_size": 512, "overlap_pct": 10, "rerank": False, "candidate_pool": 5}
DEVELOPMENT_CONFIGS = [
    BASE_CONFIG,
    {"config_id": "D1", "chunk_size": 256, "overlap_pct": 10, "rerank": False, "candidate_pool": 5},
    {"config_id": "D2", "chunk_size": 384, "overlap_pct": 10, "rerank": False, "candidate_pool": 5},
    {"config_id": "D3", "chunk_size": 512, "overlap_pct": 0, "rerank": False, "candidate_pool": 5},
    {"config_id": "D4", "chunk_size": 512, "overlap_pct": 20, "rerank": False, "candidate_pool": 5},
    {"config_id": "D5", "chunk_size": 512, "overlap_pct": 10, "rerank": True, "candidate_pool": 15},
    {"config_id": "P10", "chunk_size": 512, "overlap_pct": 10, "rerank": True, "candidate_pool": 10},
    {"config_id": "P15", "chunk_size": 512, "overlap_pct": 10, "rerank": True, "candidate_pool": 15},
    {"config_id": "P20", "chunk_size": 512, "overlap_pct": 10, "rerank": True, "candidate_pool": 20},
]


## 9. Run every pre-specified development retrieval configuration

Only the 30 development questions enter this step. The controlled comparisons are:

- D0/D1/D2 for chunk size (512/256/384 at 10% overlap, reranking off);
- D0/D3/D4 for overlap (10/0/20% at 512 tokens, reranking off);
- D0/D5 for reranking (off/on at the baseline chunk setting); and
- P10/P15/P20 for reranker candidate-pool sensitivity.

Selection is lexicographic and fixed in advance: mean Recall@5, then MRR, then Hit Rate@5, then lower median query latency. Answer generation is not used to choose retrieval settings.


In [ ]:
development_questions = questions_df[questions_df["split"] == "development"].copy()
if len(development_questions) != 30:
    raise ValueError("Development split must contain exactly 30 questions")

load_retrieval_models(load_reranker=True)
development_frames = []
for config in DEVELOPMENT_CONFIGS:
    development_frames.append(run_retrieval_config(config, development_questions))

development_retrieval_df = pd.concat(development_frames, ignore_index=True)
development_summary_df = summarise_retrieval(development_retrieval_df)
write_utf8_csv(development_retrieval_df, DIRS["development"] / "development_retrieval_runs.csv")
write_utf8_csv(development_summary_df, DIRS["development"] / "development_retrieval_summary.csv")
display(development_summary_df.sort_values(
    ["mean_recall_at_5", "mean_mrr", "mean_hit_rate_at_5", "median_latency_seconds"],
    ascending=[False, False, False, True],
))


In [ ]:
def select_best(summary: pd.DataFrame, config_ids: list[str]) -> str:
    candidates = summary[summary["config_id"].isin(config_ids)].copy()
    candidates = candidates.sort_values(
        ["mean_recall_at_5", "mean_mrr", "mean_hit_rate_at_5", "median_latency_seconds", "config_id"],
        ascending=[False, False, False, True, True], kind="stable",
    )
    if candidates.empty:
        raise ValueError(f"No development results for {config_ids}")
    return str(candidates.iloc[0]["config_id"])


CONFIG_LOOKUP = {item["config_id"]: item for item in DEVELOPMENT_CONFIGS}
best_chunk_id = select_best(development_summary_df, ["D1", "D2", "D0"])
best_overlap_id = select_best(development_summary_df, ["D3", "D0", "D4"])
best_rerank_id = select_best(development_summary_df, ["D0", "D5"])

selected_rerank = best_rerank_id == "D5"
if selected_rerank:
    best_pool_id = select_best(development_summary_df, ["P10", "P15", "P20"])
    selected_pool = CONFIG_LOOKUP[best_pool_id]["candidate_pool"]
else:
    best_pool_id = "NOT_APPLICABLE"
    selected_pool = CFG.top_k

ENHANCED_CONFIG = {
    "config_id": "D6",
    "chunk_size": CONFIG_LOOKUP[best_chunk_id]["chunk_size"],
    "overlap_pct": CONFIG_LOOKUP[best_overlap_id]["overlap_pct"],
    "rerank": selected_rerank,
    "candidate_pool": selected_pool,
}

development_d6_df = run_retrieval_config(ENHANCED_CONFIG, development_questions)
development_retrieval_df = pd.concat([development_retrieval_df, development_d6_df], ignore_index=True)
development_summary_df = summarise_retrieval(development_retrieval_df)
write_utf8_csv(development_retrieval_df, DIRS["development"] / "development_retrieval_runs_with_D6.csv")
write_utf8_csv(development_summary_df, DIRS["development"] / "development_retrieval_summary_with_D6.csv")

selection_record = {
    "selection_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "selection_split": "development only",
    "ranking_rule": "Mean Recall@5, then MRR, then Hit Rate@5, then lower median query latency",
    "best_chunk_comparison": best_chunk_id,
    "best_overlap_comparison": best_overlap_id,
    "best_reranking_comparison": best_rerank_id,
    "best_candidate_pool_comparison": best_pool_id,
    "enhanced_config": ENHANCED_CONFIG,
}
(DIRS["development"] / "development_selection.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
print(json.dumps(selection_record, indent=2))
display(development_summary_df)


## 10. Fixed generator and controlled system runner

The same Qwen generator, deterministic decoding rule, seed and answer-length limit are used for LLM-only, Basic RAG and Enhanced RAG. The two separately frozen prompt templates are system-appropriate: the RAG prompt receives evidence and requires citations, while the LLM-only prompt receives no retrieved evidence and is not asked to fabricate citations. Results are checkpointed after every response so a disconnected Colab session can resume without selective rerunning.

The functions in this section are also used by the required development sanity check. Pre-freeze sanity rows carry the explicit protocol label `PREFREEZE_DEVELOPMENT_SANITY`; held-out rows can only carry the later frozen protocol hash.


In [ ]:
generator_tokenizer = None
generator_model = None


def load_generator():
    global generator_tokenizer, generator_model
    if generator_model is not None:
        return generator_tokenizer, generator_model
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Google Colab GPU runtime before loading Qwen2.5-3B-Instruct.")
    generator_tokenizer = AutoTokenizer.from_pretrained(
        CFG.generator_model, revision=MODEL_REVISIONS["generator"], use_fast=True
    )
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    generator_model = AutoModelForCausalLM.from_pretrained(
        CFG.generator_model,
        revision=MODEL_REVISIONS["generator"],
        quantization_config=quantization,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    generator_model.eval()
    return generator_tokenizer, generator_model


def format_evidence(retrieved: list[dict[str, Any]]) -> str:
    blocks = []
    for item in retrieved:
        location = f"page {item['page']}" if str(item.get("page", "")).strip() else str(item.get("section", ""))
        header = f"[{item['document_id']} | {item['chunk_id']} | {location}]"
        blocks.append(f"{header}\n{item['text']}")
    return "\n\n".join(blocks)


def build_prompt(system_name: str, question: str, retrieved: list[dict[str, Any]] | None = None) -> str:
    if system_name == "LLM_ONLY":
        return LLM_ONLY_PROMPT.format(question=question)
    return RAG_PROMPT.format(
        refusal=REFUSAL_TEXT, question=question, evidence=format_evidence(retrieved or [])
    )


@torch.inference_mode()
def generate_answer(prompt: str) -> str:
    tokenizer, model = load_generator()
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    input_length = int(inputs["input_ids"].shape[-1])
    if input_length > CFG.max_input_tokens:
        raise ValueError(
            f"Frozen prompt contains {input_length} tokens, above the {CFG.max_input_tokens}-token limit. "
            "Do not silently truncate the evidence."
        )
    inputs = {name: value.to(model.device) for name, value in inputs.items()}
    output = model.generate(
        **inputs,
        max_new_tokens=CFG.max_new_tokens,
        do_sample=CFG.generation_do_sample,
        num_beams=CFG.generation_num_beams,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    generated = output[0][input_length:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def run_one(system_name: str, question: dict[str, Any], system_config: dict[str, Any] | None) -> dict[str, Any]:
    retrieved, retrieval_seconds = [], 0.0
    if system_name != "LLM_ONLY":
        started = time.perf_counter()
        retrieved = retrieve(
            question["question_text"], documents_df, page_map_df,
            chunk_size=int(system_config["chunk_size"]),
            overlap_pct=int(system_config["overlap_pct"]),
            rerank=bool(system_config["rerank"]),
            candidate_pool=int(system_config["candidate_pool"]),
            top_k=CFG.top_k,
        )
        retrieval_seconds = time.perf_counter() - started

    prompt = build_prompt(system_name, question["question_text"], retrieved)
    answer, generation_seconds, error = "", np.nan, ""
    status, infrastructure_rerun_used = "pending", False
    for attempt in (1, 2):
        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            started = time.perf_counter()
            answer = generate_answer(prompt)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            generation_seconds = time.perf_counter() - started
            status = "ok" if answer.strip() else "empty_model_response_incorrect"
            break
        except (RuntimeError, OSError, TimeoutError) as exc:
            error = f"{type(exc).__name__}: {exc}"
            if attempt == 1:
                infrastructure_rerun_used = True
                gc.collect()
                torch.cuda.empty_cache()
                continue
            status = "infrastructure_failure_missing"
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            status = "non_infrastructure_failure"
            break

    total = retrieval_seconds + (float(generation_seconds) if np.isfinite(generation_seconds) else 0.0)
    return {
        "run_id": str(uuid.uuid4()),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "seed": CFG.seed,
        "protocol_sha256": globals().get("FROZEN_PROTOCOL", {}).get(
            "protocol_sha256", "PREFREEZE_DEVELOPMENT_SANITY"
        ),
        "question_id": question["question_id"],
        "split": question["split"],
        "primary_category": question["primary_category"],
        "answerable": bool(question["answerable"]),
        "system_name": system_name,
        "config_json": json.dumps(system_config or {}, sort_keys=True),
        "prompt": prompt,
        "answer": answer,
        "retrieved_json": json.dumps(retrieved, ensure_ascii=False, default=str),
        "retrieval_seconds": retrieval_seconds,
        "generation_seconds": generation_seconds,
        "total_seconds": total,
        "status": status,
        "infrastructure_rerun_used": infrastructure_rerun_used,
        "error": error,
    }


def run_systems(questions: pd.DataFrame, systems: dict[str, dict[str, Any] | None], output_path: Path) -> pd.DataFrame:
    if output_path.exists():
        existing = pd.read_json(output_path, lines=True)
    else:
        existing = pd.DataFrame()
    completed = set(zip(existing.get("question_id", []), existing.get("system_name", [])))
    records = existing.to_dict("records")
    for question in questions.sort_values("question_id").to_dict("records"):
        for system_name, system_config in systems.items():
            key = (question["question_id"], system_name)
            if key in completed:
                continue
            print(f"{question['question_id']} | {system_name}")
            records.append(run_one(system_name, question, system_config))
            pd.DataFrame(records).to_json(
                output_path, orient="records", lines=True, force_ascii=False
            )
    return pd.DataFrame(records)


def warm_up_for_timing() -> dict[str, Any]:
    """Load every timed component and perform unscored warm-up work."""
    started = time.perf_counter()
    load_generator()
    warm_question = development_questions.sort_values("question_id").iloc[0]["question_text"]
    retrieve(
        warm_question, documents_df, page_map_df,
        chunk_size=BASE_CONFIG["chunk_size"], overlap_pct=BASE_CONFIG["overlap_pct"],
        rerank=BASE_CONFIG["rerank"], candidate_pool=BASE_CONFIG["candidate_pool"], top_k=CFG.top_k,
    )
    retrieve(
        warm_question, documents_df, page_map_df,
        chunk_size=ENHANCED_CONFIG["chunk_size"], overlap_pct=ENHANCED_CONFIG["overlap_pct"],
        rerank=ENHANCED_CONFIG["rerank"], candidate_pool=ENHANCED_CONFIG["candidate_pool"], top_k=CFG.top_k,
    )
    generate_answer("Warm-up only. Reply with the word ready.")
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "warm_up_seconds_excluded_from_results": time.perf_counter() - started,
        "development_question_used_for_retrieval_only": str(development_questions.sort_values("question_id").iloc[0]["question_id"]),
        "generator_prompt": "Warm-up only; not an evaluation question and not scored",
    }
    (DIRS["runs"] / "timing_warmup_record.json").write_text(
        json.dumps(record, indent=2), encoding="utf-8"
    )
    return record


## 11. Required development T4 sanity check

Before the experimental protocol can be frozen, this unscored check runs one deterministic development question from each primary category through LLM-only, Basic RAG and Enhanced RAG. It confirms that the generator, prompts, retrieval paths, quantisation and checkpoint writing operate successfully on the stated NVIDIA T4 runtime.

This check does **not** select a configuration, contribute answer-quality results or expose held-out questions. In a viva, its purpose is simply to demonstrate that the complete pipeline worked before the research protocol was locked.


In [ ]:
SANITY_OUTPUT_PATH = DIRS["runs"] / "development_t4_sanity.jsonl"
SANITY_RECORD_PATH = DIRS["development"] / "development_t4_sanity_record.json"
PROTOCOL_PATH = DIRS["protocol"] / "protocol_freeze.json"


def verify_saved_sanity() -> dict[str, Any]:
    if not SANITY_OUTPUT_PATH.exists() or not SANITY_RECORD_PATH.exists():
        raise FileNotFoundError("The required pre-freeze development sanity files are missing.")
    record = json.loads(SANITY_RECORD_PATH.read_text(encoding="utf-8"))
    if record.get("status") != "PASS":
        raise RuntimeError("The recorded development sanity check did not pass.")
    if sha256_file(SANITY_OUTPUT_PATH) != record.get("output_sha256"):
        raise ValueError("The development sanity output hash does not match its record.")
    frame = pd.read_json(SANITY_OUTPUT_PATH, lines=True)
    if len(frame) != record.get("expected_outputs") or not frame["status"].eq("ok").all():
        raise RuntimeError("The saved development sanity outputs are incomplete or contain failures.")
    return record


if PROTOCOL_PATH.exists():
    DEVELOPMENT_SANITY_RECORD = verify_saved_sanity()
    print("Existing pre-freeze T4 sanity record verified; the protocol is checked next.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Select Runtime > Change runtime type > T4 GPU before continuing.")
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" not in gpu_name.upper():
        raise RuntimeError(f"This protocol expects an NVIDIA T4 GPU; the current device is {gpu_name!r}.")

    sanity_questions = (
        development_questions.sort_values("question_id")
        .groupby("primary_category", sort=True, as_index=False)
        .head(1)
        .copy()
    )
    expected_categories = {"lookup", "multi-passage", "confusable", "unanswerable"}
    if set(sanity_questions["primary_category"]) != expected_categories:
        raise ValueError("The development sanity sample must contain one question from every category.")

    sanity_systems = {
        "LLM_ONLY": None,
        "BASIC_RAG": BASE_CONFIG,
        "ENHANCED_RAG": ENHANCED_CONFIG,
    }
    DEVELOPMENT_SANITY_DF = run_systems(sanity_questions, sanity_systems, SANITY_OUTPUT_PATH)
    expected_outputs = len(sanity_questions) * len(sanity_systems)
    if len(DEVELOPMENT_SANITY_DF) != expected_outputs:
        raise RuntimeError(
            f"Expected {expected_outputs} development sanity outputs; found {len(DEVELOPMENT_SANITY_DF)}."
        )
    failed = DEVELOPMENT_SANITY_DF[DEVELOPMENT_SANITY_DF["status"] != "ok"]
    if not failed.empty:
        display(failed[["question_id", "system_name", "status", "error"]])
        raise RuntimeError("The required development sanity check failed; the protocol was not frozen.")

    DEVELOPMENT_SANITY_RECORD = {
        "status": "PASS",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": "Unscored pre-freeze pipeline verification only; not used for configuration selection or reported answer-quality results.",
        "split": "development",
        "question_ids": sanity_questions["question_id"].tolist(),
        "categories": sorted(sanity_questions["primary_category"].tolist()),
        "systems": list(sanity_systems),
        "expected_outputs": expected_outputs,
        "output_sha256": sha256_file(SANITY_OUTPUT_PATH),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_version": torch.version.cuda,
        "gpu": gpu_name,
    }
    SANITY_RECORD_PATH.write_text(
        json.dumps(DEVELOPMENT_SANITY_RECORD, indent=2), encoding="utf-8"
    )
    print("Required development T4 sanity check: PASS")
    display(DEVELOPMENT_SANITY_DF[["question_id", "primary_category", "system_name", "status"]])


## 12. Freeze the experimental protocol after the sanity check

The protocol records immutable model revisions, tokenizer/chat-template metadata, quantisation and generation controls, the two system-appropriate prompts, the full scoring rubric, D0 and D6 configurations, development-result hashes, the required T4 sanity-check hash, the preprocessing freeze identifier and the runtime environment. It also writes an exact requirements snapshot.

If a valid protocol already exists, it is never overwritten: the notebook verifies its hash, dataset, configurations, model revisions, sanity record, requirements file and exact Python/PyTorch/CUDA/GPU/package environment. Any mismatch stops the run before held-out generation.


In [ ]:
FROZEN_PACKAGE_NAMES = [
    "pymupdf", "python-docx", "pandas", "numpy", "pyarrow", "faiss-cpu",
    "sentence-transformers", "transformers", "huggingface-hub", "accelerate",
    "bitsandbytes", "torch", "scipy", "statsmodels", "seaborn", "openpyxl", "tqdm",
]


def installed_versions(names: Iterable[str]) -> dict[str, str]:
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def system_memory_gb() -> float:
    page_size = os.sysconf("SC_PAGE_SIZE")
    physical_pages = os.sysconf("SC_PHYS_PAGES")
    return round(page_size * physical_pages / (1024 ** 3), 2)


def capture_environment(package_versions: dict[str, str] | None = None) -> dict[str, Any]:
    package_versions = package_versions or installed_versions(FROZEN_PACKAGE_NAMES)
    gpu_properties = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "cuda_version": torch.version.cuda,
        "gpu_memory_gb": round(gpu_properties.total_memory / (1024 ** 3), 2) if gpu_properties else 0,
        "system_memory_gb": system_memory_gb(),
        "packages": package_versions,
    }


def assert_runtime_matches(frozen_environment: dict[str, Any]) -> None:
    current = capture_environment()
    exact_fields = ["python", "torch", "cuda_available", "cuda_device", "cuda_version", "packages"]
    differences = {
        field: {"frozen": frozen_environment.get(field), "current": current.get(field)}
        for field in exact_fields
        if frozen_environment.get(field) != current.get(field)
    }
    if differences:
        requirements_path = DIRS["protocol"] / "requirements_frozen.txt"
        detail = json.dumps(differences, indent=2, ensure_ascii=False)
        raise RuntimeError(
            "The resumed runtime differs from the frozen experimental environment. "
            f"Differences:\n{detail}\n"
            f"Reinstall the exact packages from {requirements_path}, restart the runtime, "
            "select the same T4 GPU type and run from the top. Do not continue held-out generation."
        )


def frozen_tokenizer_metadata() -> dict[str, Any]:
    tokenizer = AutoTokenizer.from_pretrained(
        CFG.generator_model, revision=MODEL_REVISIONS["generator"], use_fast=True
    )
    chat_template = tokenizer.chat_template or ""
    return {
        "class": tokenizer.__class__.__name__,
        "resolved_revision": MODEL_REVISIONS["generator"],
        "chat_template_sha256": hashlib.sha256(chat_template.encode("utf-8")).hexdigest(),
        "chat_template": chat_template,
        "special_tokens_map": tokenizer.special_tokens_map,
    }


ANALYSIS_SPECIFICATION = {
    "primary_comparisons": ["LLM_ONLY vs BASIC_RAG", "BASIC_RAG vs ENHANCED_RAG"],
    "primary_outcome": "factual_accuracy_0_1",
    "paired_tests": {"binary": "McNemar", "ordinal": "Wilcoxon-Pratt"},
    "confidence_intervals": "question-level paired bootstrap, 10000 repetitions",
    "multiple_testing": "Holm adjustment for the four pre-specified accuracy/completeness tests",
    "claim_rates": ["corpus_support", "partial_support", "hallucination", "context_grounding", "citation_correctness"],
    "rq3": "per-RAG-system evidence-complete contingency/odds ratio and Spearman MRR-completeness association",
    "latency": "one post-warm-up measurement per response; median and IQR across held-out questions",
}


def freeze_protocol() -> dict[str, Any]:
    path = DIRS["protocol"] / "protocol_freeze.json"
    if path.exists():
        record = json.loads(path.read_text(encoding="utf-8"))
        canonical = json.dumps({k: v for k, v in record.items() if k != "protocol_sha256"}, sort_keys=True, ensure_ascii=False)
        calculated = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        if calculated != record["protocol_sha256"]:
            raise ValueError("Existing protocol freeze hash is invalid")
        if record["dataset_freeze_record_sha256"] != DATASET_FREEZE["freeze_record_sha256"]:
            raise ValueError("Existing protocol points to a different frozen dataset")
        if record["basic_config"] != BASE_CONFIG or record["enhanced_config"] != ENHANCED_CONFIG:
            raise ValueError("Current development selection differs from the frozen protocol")
        if record.get("model_revisions") != MODEL_REVISIONS:
            raise ValueError("Resolved model revisions differ from the frozen protocol")
        requirements_path = DIRS["protocol"] / "requirements_frozen.txt"
        if not requirements_path.exists() or sha256_file(requirements_path) != record.get("requirements_sha256"):
            raise ValueError("The exact frozen requirements file is missing or has changed")
        sanity_record = verify_saved_sanity()
        if record.get("development_sanity", {}).get("output_sha256") != sanity_record.get("output_sha256"):
            raise ValueError("The development sanity record differs from the frozen protocol")
        assert_runtime_matches(record["environment"])
        print("Existing protocol and exact runtime environment verified:", record["protocol_sha256"])
        return record

    sanity_record = verify_saved_sanity()
    package_versions = installed_versions(FROZEN_PACKAGE_NAMES)
    missing_packages = [name for name, version in package_versions.items() if version == "not-installed"]
    if missing_packages:
        raise RuntimeError(f"Required packages are missing: {missing_packages}")
    requirements_path = DIRS["protocol"] / "requirements_frozen.txt"
    requirements_path.write_text(
        "\n".join(f"{name}=={version}" for name, version in sorted(package_versions.items())) + "\n",
        encoding="utf-8",
    )
    rubric_path = DIRS["protocol"] / "scoring_rubric.json"
    rubric_path.write_text(json.dumps(SCORING_RUBRIC, indent=2, ensure_ascii=False), encoding="utf-8")
    examples_path = DIRS["protocol"] / "scoring_examples_development_only.csv"
    write_utf8_csv(pd.DataFrame(SCORING_EXAMPLES), examples_path)
    analysis_path = DIRS["protocol"] / "analysis_specification.json"
    analysis_path.write_text(json.dumps(ANALYSIS_SPECIFICATION, indent=2), encoding="utf-8")
    tokenizer_metadata = frozen_tokenizer_metadata()
    environment = capture_environment(package_versions)

    record = {
        "freeze_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_freeze_record_sha256": DATASET_FREEZE["freeze_record_sha256"],
        "settings": asdict(CFG),
        "model_revisions": MODEL_REVISIONS,
        "generator_tokenizer": tokenizer_metadata,
        "quantisation": {
            "load_in_4bit": True,
            "type": "nf4",
            "double_quantisation": True,
            "compute_dtype": "float16",
            "library": "bitsandbytes",
        },
        "generation": {
            "do_sample": CFG.generation_do_sample,
            "temperature": CFG.generation_temperature,
            "num_beams": CFG.generation_num_beams,
            "max_input_tokens": CFG.max_input_tokens,
            "max_new_tokens": CFG.max_new_tokens,
            "seed": CFG.seed,
        },
        "prompts": {"rag": RAG_PROMPT, "llm_only": LLM_ONLY_PROMPT, "refusal": REFUSAL_TEXT},
        "scoring_rubric_sha256": sha256_file(rubric_path),
        "scoring_examples_sha256": sha256_file(examples_path),
        "analysis_specification_sha256": sha256_file(analysis_path),
        "requirements_sha256": sha256_file(requirements_path),
        "basic_config": BASE_CONFIG,
        "enhanced_config": ENHANCED_CONFIG,
        "development_selection": selection_record,
        "development_sanity": sanity_record,
        "development_result_hashes": {
            "runs": sha256_file(DIRS["development"] / "development_retrieval_runs_with_D6.csv"),
            "summary": sha256_file(DIRS["development"] / "development_retrieval_summary_with_D6.csv"),
        },
        "heldout_systems": ["LLM_ONLY", "BASIC_RAG", "ENHANCED_RAG"],
        "failure_rule": "One logged rerun for a confirmed infrastructure exception only; empty output is incorrect; second infrastructure failure remains missing.",
        "manual_evaluation": "Researcher-only, response-order randomised and system-blinded; no independent reviewer.",
        "environment": environment,
    }
    canonical = json.dumps(record, sort_keys=True, ensure_ascii=False)
    record["protocol_sha256"] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    (DIRS["protocol"] / "PROTOCOL_FROZEN.txt").write_text(
        f"Protocol SHA-256: {record['protocol_sha256']}\n", encoding="utf-8"
    )
    print("Protocol frozen:", record["protocol_sha256"])
    return record


FROZEN_PROTOCOL = freeze_protocol()


## 13. Held-out gate and final controlled runs

Do not pass this gate until the dataset freeze, required T4 development sanity check and experimental-protocol freeze have all been verified. No further retrieval, prompt, scoring or analysis choices may be made after confirmation. The final evaluation runs the same 90 held-out questions through LLM-only, Basic RAG and Enhanced RAG, producing exactly 270 outputs.


In [ ]:
HELDOUT_CONFIRMATION = input(
    "To run the held-out evaluation, type exactly: I CONFIRM THE PROTOCOL IS FROZEN\n"
).strip()

if HELDOUT_CONFIRMATION != "I CONFIRM THE PROTOCOL IS FROZEN":
    print("Held-out generation not started. This is a deliberate protocol safeguard.")
    heldout_results_df = pd.DataFrame()
else:
    verify_frozen_folder(DIRS["frozen"])
    DEVELOPMENT_SANITY_RECORD = verify_saved_sanity()
    FROZEN_PROTOCOL = freeze_protocol()
    WARMUP_RECORD = warm_up_for_timing()
    print("Model/index warm-up completed; warm-up time is excluded from latency results.")
    heldout_questions = questions_df[questions_df["split"] == "held-out"].copy()
    if len(heldout_questions) != 90:
        raise ValueError("Held-out split must contain exactly 90 questions")
    final_systems = {
        "LLM_ONLY": None,
        "BASIC_RAG": BASE_CONFIG,
        "ENHANCED_RAG": ENHANCED_CONFIG,
    }
    heldout_results_df = run_systems(
        heldout_questions, final_systems, DIRS["runs"] / "heldout_results.jsonl"
    )
    write_utf8_csv(heldout_results_df, DIRS["runs"] / "heldout_results.csv")
    expected_pairs = 90 * 3
    if len(heldout_results_df) != expected_pairs:
        raise ValueError(f"Expected {expected_pairs} held-out system outputs; found {len(heldout_results_df)}")
    print(heldout_results_df.groupby(["system_name", "status"]).size())


## 14. Automatic retrieval, citation and refusal checks

These objective fields supplement—rather than replace—the researcher’s factual-accuracy and completeness scoring. Citation precision checks only whether cited chunk identifiers came from the retrieved context; it does not by itself establish that the prose is entailed.


In [ ]:
CITATION_PATTERN = re.compile(r"UEL\d{3}::c\d{4}")


def automatic_output_metrics(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in results.to_dict("records"):
        retrieved = json.loads(row["retrieved_json"]) if row.get("retrieved_json") else []
        retrieved_ids = {item["chunk_id"] for item in retrieved}
        cited_ids = set(CITATION_PATTERN.findall(str(row.get("answer", ""))))
        valid_citations = cited_ids & retrieved_ids
        retrieval_metrics = (
            retrieval_metrics_for_question(retrieved, row["question_id"], evidence_df)
            if row["system_name"] != "LLM_ONLY" else
            {"hit_rate_at_5": np.nan, "recall_at_5": np.nan, "mrr": np.nan,
             "matched_evidence_count": 0, "gold_evidence_count": 0, "matched_evidence_ids": []}
        )
        answer_normal = " ".join(str(row.get("answer", "")).casefold().split())
        exact_refusal = " ".join(REFUSAL_TEXT.casefold().split()) in answer_normal
        rows.append({
            "run_id": row["run_id"],
            "question_id": row["question_id"],
            "system_name": row["system_name"],
            "primary_category": row["primary_category"],
            "answerable": row["answerable"],
            "status": row["status"],
            "hit_rate_at_5": retrieval_metrics["hit_rate_at_5"],
            "recall_at_5": retrieval_metrics["recall_at_5"],
            "mrr": retrieval_metrics["mrr"],
            "matched_evidence_ids": json.dumps(retrieval_metrics["matched_evidence_ids"]),
            "citation_count": len(cited_ids),
            "valid_citation_count": len(valid_citations),
            "citation_retrieval_precision": len(valid_citations) / len(cited_ids) if cited_ids else np.nan,
            "all_citations_retrieved": bool(cited_ids) and cited_ids <= retrieved_ids,
            "exact_corpus_refusal": exact_refusal,
            "retrieval_seconds": row["retrieval_seconds"],
            "generation_seconds": row["generation_seconds"],
            "total_seconds": row["total_seconds"],
        })
    return pd.DataFrame(rows)


if not heldout_results_df.empty:
    objective_metrics_df = automatic_output_metrics(heldout_results_df)
    write_utf8_csv(objective_metrics_df, DIRS["evaluation"] / "heldout_objective_metrics.csv")
    objective_summary_df = objective_metrics_df.groupby("system_name", as_index=False).agg(
        hit_rate_at_5=("hit_rate_at_5", "mean"),
        recall_at_5=("recall_at_5", "mean"),
        mrr=("mrr", "mean"),
        citation_retrieval_precision=("citation_retrieval_precision", "mean"),
        exact_corpus_refusal=("exact_corpus_refusal", "mean"),
        median_total_seconds=("total_seconds", "median"),
        total_seconds_q1=("total_seconds", lambda values: values.quantile(0.25)),
        total_seconds_q3=("total_seconds", lambda values: values.quantile(0.75)),
    )
    objective_summary_df["total_seconds_iqr"] = objective_summary_df["total_seconds_q3"] - objective_summary_df["total_seconds_q1"]
    write_utf8_csv(objective_summary_df, DIRS["evaluation"] / "heldout_objective_summary.csv")
    display(objective_summary_df)


## 15. Researcher-only blinded scoring files

Primary scoring and retrieved-context review are separated to preserve blinding. The primary sheet contains all 270 answers, the frozen reference material and claim-count fields, but no system label or retrieved context. A separately shuffled sheet contains the 180 RAG answers and score-free retrieved passages, while still hiding Basic/Enhanced labels. Similarity and reranker scores are deliberately removed because they could reveal the system.

Use the frozen rubric and DEV-only examples in the workbook. Atomic claim counts generate corpus-support, partial-support and hallucination rates for all systems. The RAG sheet separately generates retrieved-context grounding, context hallucination, citation correctness and citation coverage. Complete both sheets without opening either blinding key.


In [ ]:
PRIMARY_SCORE_COLUMNS = [
    "factual_accuracy_0_1", "completeness_0_2", "response_refusal_0_1",
    "correct_refusal_0_1", "false_answer_0_1",
    "evaluable_claims_count", "fully_corpus_supported_claims",
    "partially_corpus_supported_claims", "unsupported_claims", "contradicted_claims",
    "ambiguous_claims_count", "nonfactual_statements_count", "researcher_notes",
]
RAG_SCORE_COLUMNS = [
    "context_evaluable_claims_count", "fully_context_supported_claims",
    "partially_context_supported_claims", "unsupported_by_context_claims",
    "contradicted_by_context_claims", "context_ambiguous_claims_count",
    "citations_total", "citations_supporting_claims",
    "citations_incorrect_or_unsupported", "evaluable_claims_with_citation",
    "grounding_notes",
]


def score_free_retrieved_json(value: str) -> str:
    retrieved = json.loads(value) if value else []
    permitted = [
        {
            "rank": item.get("rank"), "chunk_id": item.get("chunk_id"),
            "document_id": item.get("document_id"), "page": item.get("page"),
            "section": item.get("section"), "text": item.get("text"),
        }
        for item in retrieved
    ]
    return json.dumps(permitted, ensure_ascii=False)


def rubric_rows() -> pd.DataFrame:
    rows = []
    for section, content in SCORING_RUBRIC.items():
        if isinstance(content, dict):
            for field, rule in content.items():
                rows.append({"section": section, "field": field, "instruction": rule})
        else:
            rows.append({"section": "general", "field": section, "instruction": content})
    return pd.DataFrame(rows)


def style_scoring_workbook(path: Path) -> None:
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Font, PatternFill

    workbook = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    for sheet in workbook.worksheets:
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for cell in sheet[1]:
            cell.fill = header_fill
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="center")
        for row in sheet.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
        for column_cells in sheet.columns:
            letter = column_cells[0].column_letter
            header = str(column_cells[0].value or "")
            if header in {"question_text", "reference_answer", "evidence_text", "retrieved_context", "answer", "instruction", "response", "decision"}:
                width = 55
            elif header in {"source_locations", "required_points", "validation_notes", "researcher_notes", "grounding_notes"}:
                width = 38
            else:
                width = min(24, max(12, len(header) + 2))
            sheet.column_dimensions[letter].width = width
    workbook.save(path)


def create_blinded_scoring_files(results: pd.DataFrame, questions: pd.DataFrame):
    question_fields = questions[[
        "question_id", "question_text", "reference_answer", "required_points",
        "evidence_text", "source_locations", "validation_notes",
        "primary_category", "answerable",
    ]]
    merged = results.merge(question_fields, on=["question_id", "primary_category", "answerable"], how="left", validate="many_to_one")
    rng = np.random.default_rng(CFG.seed)
    primary = merged.iloc[rng.permutation(len(merged))].reset_index(drop=True)
    primary["blind_output_id"] = [f"PRIMARY_{index:04d}" for index in range(1, len(primary) + 1)]
    primary_key = primary[["blind_output_id", "run_id", "system_name"]].copy()
    primary_scoring = primary[[
        "blind_output_id", "question_id", "primary_category", "answerable",
        "question_text", "reference_answer", "required_points", "evidence_text",
        "source_locations", "validation_notes", "answer", "status",
    ]].copy()
    for column in PRIMARY_SCORE_COLUMNS:
        primary_scoring[column] = ""

    rag = merged[merged["system_name"] != "LLM_ONLY"].copy()
    rag = rag.iloc[rng.permutation(len(rag))].reset_index(drop=True)
    rag["grounding_blind_id"] = [f"GROUND_{index:04d}" for index in range(1, len(rag) + 1)]
    grounding_key = rag[["grounding_blind_id", "run_id", "system_name"]].copy()
    rag["retrieved_context"] = rag["retrieved_json"].map(score_free_retrieved_json)
    grounding_scoring = rag[[
        "grounding_blind_id", "question_id", "question_text", "answer", "retrieved_context", "status",
    ]].copy()
    for column in RAG_SCORE_COLUMNS:
        grounding_scoring[column] = ""

    primary_path = DIRS["evaluation"] / "researcher_blinded_primary_scoring.csv"
    grounding_path = DIRS["evaluation"] / "researcher_blinded_RAG_grounding.csv"
    primary_key_path = DIRS["evaluation"] / "DO_NOT_OPEN_primary_blinding_key.csv"
    grounding_key_path = DIRS["evaluation"] / "DO_NOT_OPEN_grounding_blinding_key.csv"
    write_utf8_csv(primary_scoring, primary_path)
    write_utf8_csv(grounding_scoring, grounding_path)
    write_utf8_csv(primary_key, primary_key_path)
    write_utf8_csv(grounding_key, grounding_key_path)

    workbook_path = DIRS["evaluation"] / "researcher_blinded_scoring.xlsx"
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        primary_scoring.to_excel(writer, sheet_name="Primary scoring", index=False)
        grounding_scoring.to_excel(writer, sheet_name="RAG grounding", index=False)
        rubric_rows().to_excel(writer, sheet_name="Rubric", index=False)
        pd.DataFrame(SCORING_EXAMPLES).to_excel(writer, sheet_name="DEV examples", index=False)
        pd.DataFrame({
            "step": [1, 2, 3, 4],
            "instruction": [
                "Score every row in Primary scoring without opening either blinding key.",
                "Score every row in RAG grounding using only the score-free retrieved context.",
                "Save a completed copy, set COMPLETED_SCORING_WORKBOOK to its path, and run validation.",
                "Do not change the frozen rubric after viewing system identities or aggregate results.",
            ],
        }).to_excel(writer, sheet_name="Read me", index=False)
    style_scoring_workbook(workbook_path)
    print("Primary scoring CSV:", primary_path)
    print("RAG grounding CSV:", grounding_path)
    print("Scoring workbook:", workbook_path)
    print("Keep both blinding keys closed until both scoring sheets are validated and locked.")
    return primary_path, grounding_path, primary_key_path, grounding_key_path, workbook_path


if not heldout_results_df.empty:
    (PRIMARY_SCORING_PATH, GROUNDING_SCORING_PATH, PRIMARY_KEY_PATH,
     GROUNDING_KEY_PATH, SCORING_WORKBOOK_PATH) = create_blinded_scoring_files(
        heldout_results_df, questions_df
    )


In [ ]:
def read_scoring_file(path: Path, sheet_name: str) -> pd.DataFrame:
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path, sheet_name=sheet_name, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def validate_and_lock_scoring(primary_path: Path, grounding_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    primary = read_scoring_file(primary_path, "Primary scoring")
    grounding = read_scoring_file(grounding_path, "RAG grounding")
    missing_primary = {"blind_output_id", *PRIMARY_SCORE_COLUMNS} - set(primary.columns)
    missing_grounding = {"grounding_blind_id", *RAG_SCORE_COLUMNS} - set(grounding.columns)
    if missing_primary or missing_grounding:
        raise ValueError(f"Missing primary columns={sorted(missing_primary)}; grounding columns={sorted(missing_grounding)}")
    if primary["blind_output_id"].duplicated().any() or len(primary) != 270:
        raise ValueError("Primary scoring must contain 270 unique blinded outputs")
    if grounding["grounding_blind_id"].duplicated().any() or len(grounding) != 180:
        raise ValueError("RAG grounding scoring must contain 180 unique blinded outputs")

    primary_binary_columns = [
        "factual_accuracy_0_1", "response_refusal_0_1", "correct_refusal_0_1", "false_answer_0_1"
    ]
    for column in primary_binary_columns:
        values = pd.to_numeric(primary[column], errors="coerce")
        if values.isna().any() or not values.isin([0, 1]).all():
            raise ValueError(f"{column} must be complete and contain only 0 or 1")
        primary[column] = values.astype(int)
    completeness = pd.to_numeric(primary["completeness_0_2"], errors="coerce")
    if completeness.isna().any() or not completeness.isin([0, 1, 2]).all():
        raise ValueError("completeness_0_2 must be complete and contain 0, 1 or 2")
    primary["completeness_0_2"] = completeness.astype(int)

    primary_count_columns = [column for column in PRIMARY_SCORE_COLUMNS if column.endswith("_count") or column.endswith("_claims")]
    for column in primary_count_columns:
        values = pd.to_numeric(primary[column], errors="coerce")
        if values.isna().any() or (values < 0).any() or not np.equal(values, np.floor(values)).all():
            raise ValueError(f"{column} must contain complete non-negative integer counts")
        primary[column] = values.astype(int)
    corpus_sum = primary[[
        "fully_corpus_supported_claims", "partially_corpus_supported_claims",
        "unsupported_claims", "contradicted_claims",
    ]].sum(axis=1)
    if not corpus_sum.equals(primary["evaluable_claims_count"]):
        raise ValueError("Primary claim-label counts must sum to evaluable_claims_count")
    answerable_flags = primary["answerable"].map(parse_bool)
    if (primary.loc[answerable_flags, "correct_refusal_0_1"] != 0).any():
        raise ValueError("correct_refusal_0_1 must be 0 for answerable questions")

    rag_count_columns = [column for column in RAG_SCORE_COLUMNS if column != "grounding_notes"]
    for column in rag_count_columns:
        values = pd.to_numeric(grounding[column], errors="coerce")
        if values.isna().any() or (values < 0).any() or not np.equal(values, np.floor(values)).all():
            raise ValueError(f"{column} must contain complete non-negative integer counts")
        grounding[column] = values.astype(int)
    context_sum = grounding[[
        "fully_context_supported_claims", "partially_context_supported_claims",
        "unsupported_by_context_claims", "contradicted_by_context_claims",
    ]].sum(axis=1)
    if not context_sum.equals(grounding["context_evaluable_claims_count"]):
        raise ValueError("RAG context claim-label counts must sum to context_evaluable_claims_count")
    citation_sum = grounding[["citations_supporting_claims", "citations_incorrect_or_unsupported"]].sum(axis=1)
    if not citation_sum.equals(grounding["citations_total"]):
        raise ValueError("Citation decision counts must sum to citations_total")
    if (grounding["evaluable_claims_with_citation"] > grounding["context_evaluable_claims_count"]).any():
        raise ValueError("evaluable_claims_with_citation cannot exceed context_evaluable_claims_count")

    primary_locked_path = DIRS["evaluation"] / "researcher_primary_scoring_LOCKED.csv"
    grounding_locked_path = DIRS["evaluation"] / "researcher_RAG_grounding_LOCKED.csv"
    write_utf8_csv(primary, primary_locked_path)
    write_utf8_csv(grounding, grounding_locked_path)
    lock = {
        "locked_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "primary_file": primary_locked_path.name,
        "primary_sha256": sha256_file(primary_locked_path),
        "primary_rows": len(primary),
        "grounding_file": grounding_locked_path.name,
        "grounding_sha256": sha256_file(grounding_locked_path),
        "grounding_rows": len(grounding),
        "scoring_rubric_sha256": FROZEN_PROTOCOL["scoring_rubric_sha256"],
        "evaluator": "researcher",
        "independent_reviewer": False,
    }
    (DIRS["evaluation"] / "researcher_scoring_lock.json").write_text(
        json.dumps(lock, indent=2), encoding="utf-8"
    )
    print("Primary scoring locked:", lock["primary_sha256"])
    print("RAG grounding scoring locked:", lock["grounding_sha256"])
    return primary, grounding, lock


# Optional manual-scoring stage.
# In the original dissertation run, this workbook was completed and reviewed
# before the final statistical analysis.
#
# For a fresh public/portfolio rerun, the completed workbook may not exist.

completed_candidate = (
    DIRS["evaluation"] / "researcher_blinded_scoring_completed.xlsx"
)

if completed_candidate.exists():

    COMPLETED_SCORING_WORKBOOK = completed_candidate

    locked_primary_df, locked_grounding_df, SCORING_LOCK = (
        validate_and_lock_scoring(
            COMPLETED_SCORING_WORKBOOK,
            COMPLETED_SCORING_WORKBOOK,
        )
    )

    print("✅ Completed scoring workbook validated and locked.")

else:

    COMPLETED_SCORING_WORKBOOK = None

    print(
        "ℹ️ Completed researcher scoring workbook was not found.\n"
        "The manual scoring stage is being skipped for this fresh portfolio rerun.\n"
        "Use the committed final dissertation results for the completed experiment."
    )

In [ ]:
def unblind_locked_scores(
    locked_primary: pd.DataFrame,
    locked_grounding: pd.DataFrame,
    results: pd.DataFrame,
    primary_key_path: Path,
    grounding_key_path: Path,
    objective_metrics: pd.DataFrame,
) -> pd.DataFrame:
    lock_path = DIRS["evaluation"] / "researcher_scoring_lock.json"
    lock = json.loads(lock_path.read_text(encoding="utf-8"))
    primary_locked_path = DIRS["evaluation"] / lock["primary_file"]
    grounding_locked_path = DIRS["evaluation"] / lock["grounding_file"]
    if sha256_file(primary_locked_path) != lock["primary_sha256"] or sha256_file(grounding_locked_path) != lock["grounding_sha256"]:
        raise ValueError("A locked scoring file hash has changed")
    primary_key = pd.read_csv(primary_key_path, dtype=str, keep_default_na=False)
    grounding_key = pd.read_csv(grounding_key_path, dtype=str, keep_default_na=False)
    scored = locked_primary.merge(primary_key, on="blind_output_id", how="left", validate="one_to_one")
    grounding = locked_grounding.merge(
        grounding_key, on="grounding_blind_id", how="left", validate="one_to_one"
    )[["run_id", *RAG_SCORE_COLUMNS]]
    scored = scored.merge(grounding, on="run_id", how="left", validate="one_to_one")
    llm_only = scored["system_name"] == "LLM_ONLY"
    rag_numeric = [column for column in RAG_SCORE_COLUMNS if column != "grounding_notes"]
    if scored.loc[~llm_only, rag_numeric].isna().any().any():
        raise ValueError("Grounding scores are missing for one or more RAG outputs")
    run_fields = results[[
        "run_id", "system_name", "question_id", "primary_category", "answerable",
        "retrieval_seconds", "generation_seconds", "total_seconds", "status",
    ]].copy()
    final = scored.merge(
        run_fields, on=["run_id", "system_name", "question_id"], how="left",
        validate="one_to_one", suffixes=("", "_run"),
    )
    if final["system_name"].isna().any() or len(final) != 270:
        raise ValueError("Unblinding failed to recover all 270 outputs")

    objective_fields = objective_metrics.drop(columns=[
        "question_id", "system_name", "primary_category", "answerable", "status",
        "retrieval_seconds", "generation_seconds", "total_seconds",
    ], errors="ignore")
    final = final.merge(objective_fields, on="run_id", how="left", validate="one_to_one")

    def safe_rate(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
        return numerator.astype(float).div(denominator.replace(0, np.nan).astype(float))

    final["corpus_support_rate"] = safe_rate(final["fully_corpus_supported_claims"], final["evaluable_claims_count"])
    final["partial_support_rate"] = safe_rate(final["partially_corpus_supported_claims"], final["evaluable_claims_count"])
    final["hallucination_rate"] = safe_rate(
        final["unsupported_claims"] + final["contradicted_claims"], final["evaluable_claims_count"]
    )
    for column in rag_numeric:
        final.loc[llm_only, column] = np.nan
    final["context_grounding_rate"] = safe_rate(final["fully_context_supported_claims"], final["context_evaluable_claims_count"])
    final["context_partial_support_rate"] = safe_rate(final["partially_context_supported_claims"], final["context_evaluable_claims_count"])
    final["context_hallucination_rate"] = safe_rate(
        final["unsupported_by_context_claims"] + final["contradicted_by_context_claims"],
        final["context_evaluable_claims_count"],
    )
    final["citation_correctness_rate"] = safe_rate(final["citations_supporting_claims"], final["citations_total"])
    final["citation_coverage_rate"] = safe_rate(final["evaluable_claims_with_citation"], final["context_evaluable_claims_count"])
    write_utf8_csv(final, DIRS["evaluation"] / "heldout_researcher_scored_unblinded.csv")
    return final


if COMPLETED_SCORING_WORKBOOK is not None:
    final_scored_df = unblind_locked_scores(
        locked_primary_df, locked_grounding_df, heldout_results_df,
        PRIMARY_KEY_PATH, GROUNDING_KEY_PATH, objective_metrics_df,
    )


## 16. Paired analysis, claim-level summaries and RQ3

The primary paired comparisons are LLM-only vs Basic RAG and Basic RAG vs Enhanced RAG. Binary accuracy uses McNemar's test; completeness uses the paired Wilcoxon signed-rank test with Pratt handling of zero differences. Bootstrap confidence intervals resample question IDs, and Holm adjustment is applied to the family of four pre-specified accuracy/completeness tests. Claim-level measures are aggregated within answers before system summaries.

RQ3 is analysed separately for Basic and Enhanced RAG so that two outputs for the same question are not incorrectly treated as independent. It compares factual accuracy when all required evidence was retrieved versus when evidence was incomplete, with a continuity-corrected odds ratio and confidence interval when needed. Spearman's rank correlation describes the association between MRR and completeness.


In [ ]:
def paired_wide(frame: pd.DataFrame, outcome: str, system_a: str, system_b: str) -> pd.DataFrame:
    subset = frame[frame["system_name"].isin([system_a, system_b])]
    wide = subset.pivot(index="question_id", columns="system_name", values=outcome)
    return wide[[system_a, system_b]].dropna()


def paired_bootstrap(wide: pd.DataFrame, system_a: str, system_b: str) -> dict[str, float]:
    if wide.empty:
        return {"difference_b_minus_a": np.nan, "ci95_low": np.nan, "ci95_high": np.nan, "n_pairs": 0}
    differences = wide[system_b].astype(float).to_numpy() - wide[system_a].astype(float).to_numpy()
    rng = np.random.default_rng(CFG.seed)
    samples = np.empty(CFG.bootstrap_repetitions)
    for index in range(CFG.bootstrap_repetitions):
        samples[index] = rng.choice(differences, size=len(differences), replace=True).mean()
    return {
        "difference_b_minus_a": float(differences.mean()),
        "ci95_low": float(np.quantile(samples, 0.025)),
        "ci95_high": float(np.quantile(samples, 0.975)),
        "n_pairs": len(differences),
    }


def mcnemar_test(frame: pd.DataFrame, outcome: str, system_a: str, system_b: str) -> dict[str, Any]:
    wide = paired_wide(frame, outcome, system_a, system_b).astype(int)
    table = pd.crosstab(wide[system_a], wide[system_b]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    discordant = int(table.loc[0, 1] + table.loc[1, 0])
    exact = discordant <= 25
    test = mcnemar(table.to_numpy(), exact=exact, correction=not exact)
    return {
        "outcome": outcome, "test": "McNemar", "system_a": system_a, "system_b": system_b,
        "statistic": float(test.statistic), "p_value": float(test.pvalue),
        "exact": exact, "discordant_pairs": discordant, **paired_bootstrap(wide, system_a, system_b),
    }


def wilcoxon_test(frame: pd.DataFrame, outcome: str, system_a: str, system_b: str) -> dict[str, Any]:
    wide = paired_wide(frame, outcome, system_a, system_b).astype(float)
    differences = wide[system_b].to_numpy() - wide[system_a].to_numpy()
    if np.all(differences == 0):
        statistic, p_value = 0.0, 1.0
    else:
        test = scipy.stats.wilcoxon(
            wide[system_b], wide[system_a], zero_method="pratt", alternative="two-sided", method="auto"
        )
        statistic, p_value = float(test.statistic), float(test.pvalue)
    return {
        "outcome": outcome, "test": "Wilcoxon-Pratt", "system_a": system_a, "system_b": system_b,
        "statistic": statistic, "p_value": p_value, **paired_bootstrap(wide, system_a, system_b),
    }


def run_primary_statistics(final_scored: pd.DataFrame) -> pd.DataFrame:
    results = []
    for system_a, system_b in [("LLM_ONLY", "BASIC_RAG"), ("BASIC_RAG", "ENHANCED_RAG")]:
        results.append(mcnemar_test(final_scored, "factual_accuracy_0_1", system_a, system_b))
        results.append(wilcoxon_test(final_scored, "completeness_0_2", system_a, system_b))
    table = pd.DataFrame(results)
    table["holm_adjusted_p"] = multipletests(table["p_value"], method="holm")[1]
    return table


def divide_or_nan(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else np.nan


def score_summaries(final_scored: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for system_name, group in final_scored.groupby("system_name"):
        unanswerable = group[group["primary_category"] == "unanswerable"]
        evaluable = group["evaluable_claims_count"].sum()
        record = {
            "system_name": system_name,
            "outputs": group["question_id"].nunique(),
            "factual_accuracy": group["factual_accuracy_0_1"].mean(),
            "mean_completeness": group["completeness_0_2"].mean(),
            "corpus_support_rate": divide_or_nan(group["fully_corpus_supported_claims"].sum(), evaluable),
            "partial_support_rate": divide_or_nan(group["partially_corpus_supported_claims"].sum(), evaluable),
            "hallucination_rate": divide_or_nan((group["unsupported_claims"] + group["contradicted_claims"]).sum(), evaluable),
            "correct_refusal_rate": unanswerable["correct_refusal_0_1"].mean(),
            "false_answer_rate": unanswerable["false_answer_0_1"].mean(),
            "median_latency_seconds": group["total_seconds"].median(),
            "latency_iqr_seconds": group["total_seconds"].quantile(0.75) - group["total_seconds"].quantile(0.25),
        }
        if system_name != "LLM_ONLY":
            context_evaluable = group["context_evaluable_claims_count"].sum()
            citations = group["citations_total"].sum()
            record.update({
                "context_grounding_rate": divide_or_nan(group["fully_context_supported_claims"].sum(), context_evaluable),
                "context_partial_support_rate": divide_or_nan(group["partially_context_supported_claims"].sum(), context_evaluable),
                "context_hallucination_rate": divide_or_nan((group["unsupported_by_context_claims"] + group["contradicted_by_context_claims"]).sum(), context_evaluable),
                "citation_correctness_rate": divide_or_nan(group["citations_supporting_claims"].sum(), citations),
                "citation_coverage_rate": divide_or_nan(group["evaluable_claims_with_citation"].sum(), context_evaluable),
                "retrieval_hit_rate_at_5": group["hit_rate_at_5"].mean(),
                "retrieval_recall_at_5": group["recall_at_5"].mean(),
                "retrieval_mrr": group["mrr"].mean(),
            })
        rows.append(record)
    overall = pd.DataFrame(rows)
    category = final_scored.groupby(["system_name", "primary_category"], as_index=False).agg(
        factual_accuracy=("factual_accuracy_0_1", "mean"),
        mean_completeness=("completeness_0_2", "mean"),
        questions=("question_id", "count"),
    )
    macro = category.groupby("system_name", as_index=False)["factual_accuracy"].mean().rename(
        columns={"factual_accuracy": "macro_category_accuracy"}
    )
    overall = overall.merge(macro, on="system_name", how="left")
    return overall, category


def secondary_paired_effects(final_scored: pd.DataFrame) -> pd.DataFrame:
    comparisons = [
        ("LLM_ONLY", "BASIC_RAG", "corpus_support_rate"),
        ("LLM_ONLY", "BASIC_RAG", "hallucination_rate"),
        ("BASIC_RAG", "ENHANCED_RAG", "corpus_support_rate"),
        ("BASIC_RAG", "ENHANCED_RAG", "hallucination_rate"),
        ("BASIC_RAG", "ENHANCED_RAG", "context_grounding_rate"),
        ("BASIC_RAG", "ENHANCED_RAG", "context_hallucination_rate"),
        ("BASIC_RAG", "ENHANCED_RAG", "citation_correctness_rate"),
    ]
    rows = []
    for system_a, system_b, outcome in comparisons:
        wide = paired_wide(final_scored, outcome, system_a, system_b).astype(float)
        rows.append({"system_a": system_a, "system_b": system_b, "outcome": outcome, **paired_bootstrap(wide, system_a, system_b)})
    return pd.DataFrame(rows)


def odds_ratio_and_ci(table: np.ndarray) -> dict[str, float]:
    cells = table.astype(float)
    correction_used = bool((cells == 0).any())
    if correction_used:
        cells = cells + 0.5
    # Rows: evidence incomplete/complete. Columns: inaccurate/accurate.
    incomplete_incorrect, incomplete_correct = cells[0, 0], cells[0, 1]
    complete_incorrect, complete_correct = cells[1, 0], cells[1, 1]
    odds_ratio = (complete_correct * incomplete_incorrect) / (complete_incorrect * incomplete_correct)
    standard_error = math.sqrt(sum(1 / value for value in cells.ravel()))
    log_or = math.log(odds_ratio)
    return {
        "odds_ratio_accuracy_complete_vs_incomplete": float(odds_ratio),
        "ci95_low": float(math.exp(log_or - 1.96 * standard_error)),
        "ci95_high": float(math.exp(log_or + 1.96 * standard_error)),
        "continuity_correction_used": correction_used,
    }


def run_rq3_analysis(final_scored: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rag = final_scored[(final_scored["system_name"] != "LLM_ONLY") & final_scored["answerable"].map(parse_bool)].copy()
    rag["all_required_evidence_retrieved"] = np.isclose(rag["recall_at_5"].astype(float), 1.0)
    contingency_rows, odds_rows, correlation_rows = [], [], []
    for system_name, group in rag.groupby("system_name"):
        table_frame = pd.crosstab(
            group["all_required_evidence_retrieved"], group["factual_accuracy_0_1"]
        ).reindex(index=[False, True], columns=[0, 1], fill_value=0)
        table = table_frame.to_numpy()
        for retrieved_label, retrieved_flag in [("incomplete", False), ("complete", True)]:
            contingency_rows.append({
                "system_name": system_name,
                "evidence_retrieval": retrieved_label,
                "inaccurate": int(table_frame.loc[retrieved_flag, 0]),
                "accurate": int(table_frame.loc[retrieved_flag, 1]),
            })
        fisher = scipy.stats.fisher_exact(table)
        odds_rows.append({
            "system_name": system_name,
            "n_answerable": len(group),
            "fisher_exact_p": float(fisher.pvalue),
            **odds_ratio_and_ci(table),
        })
        valid = group[["mrr", "completeness_0_2"]].dropna().astype(float)
        if valid["mrr"].nunique() < 2 or valid["completeness_0_2"].nunique() < 2:
            rho, p_value = np.nan, np.nan
        else:
            rho, p_value = scipy.stats.spearmanr(valid["mrr"], valid["completeness_0_2"])
        correlation_rows.append({
            "system_name": system_name, "n": len(valid),
            "spearman_rho_mrr_vs_completeness": float(rho) if np.isfinite(rho) else np.nan,
            "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
        })
    return pd.DataFrame(contingency_rows), pd.DataFrame(odds_rows), pd.DataFrame(correlation_rows)


if COMPLETED_SCORING_WORKBOOK is not None:
    primary_statistics_df = run_primary_statistics(final_scored_df)
    overall_summary_df, category_summary_df = score_summaries(final_scored_df)
    secondary_effects_df = secondary_paired_effects(final_scored_df)
    rq3_contingency_df, rq3_odds_df, rq3_correlation_df = run_rq3_analysis(final_scored_df)
    write_utf8_csv(primary_statistics_df, DIRS["evaluation"] / "primary_statistical_tests.csv")
    write_utf8_csv(overall_summary_df, DIRS["evaluation"] / "heldout_system_summary.csv")
    write_utf8_csv(category_summary_df, DIRS["evaluation"] / "heldout_category_summary.csv")
    write_utf8_csv(secondary_effects_df, DIRS["evaluation"] / "secondary_paired_effects.csv")
    write_utf8_csv(rq3_contingency_df, DIRS["evaluation"] / "rq3_retrieval_accuracy_contingency.csv")
    write_utf8_csv(rq3_odds_df, DIRS["evaluation"] / "rq3_odds_ratios.csv")
    write_utf8_csv(rq3_correlation_df, DIRS["evaluation"] / "rq3_mrr_completeness_spearman.csv")
    display(primary_statistics_df)
    display(overall_summary_df)
    display(rq3_odds_df)
    display(rq3_correlation_df)


## 17. Figures and presentation outputs

These figures are generated only from saved numeric tables after scoring is locked and unblinded. They are suitable for the dissertation and viva, while the CSV files remain the authoritative results.


In [ ]:
# This cell contains presentation figures only; all numeric source tables are saved separately.
SYSTEM_ORDER = ["LLM_ONLY", "BASIC_RAG", "ENHANCED_RAG"]
SYSTEM_LABELS = {"LLM_ONLY": "LLM-only", "BASIC_RAG": "Basic RAG", "ENHANCED_RAG": "Enhanced RAG"}
PALETTE = {"LLM_ONLY": "#777777", "BASIC_RAG": "#377eb8", "ENHANCED_RAG": "#4daf4a"}


def save_figure(filename: str) -> None:
    path = DIRS["figures"] / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


def plot_development_retrieval(summary: pd.DataFrame) -> None:
    long = summary.melt(
        id_vars="config_id", value_vars=["mean_recall_at_5", "mean_mrr", "mean_hit_rate_at_5"],
        var_name="metric", value_name="score",
    )
    plt.figure(figsize=(11, 5))
    sns.barplot(data=long, x="config_id", y="score", hue="metric")
    plt.ylim(0, 1)
    plt.title("Development retrieval performance")
    plt.xlabel("Controlled configuration")
    save_figure("development_retrieval.png")


def plot_heldout_results(overall: pd.DataFrame, category: pd.DataFrame) -> None:
    order = [item for item in SYSTEM_ORDER if item in set(overall["system_name"])]
    plot_frame = overall.set_index("system_name").loc[order].reset_index()
    plt.figure(figsize=(7, 4.5))
    bars = plt.bar(
        plot_frame["system_name"].map(SYSTEM_LABELS), plot_frame["factual_accuracy"] * 100,
        color=[PALETTE[item] for item in plot_frame["system_name"]],
    )
    plt.ylim(0, 100)
    plt.ylabel("Factual accuracy (%)")
    plt.title("Held-out factual accuracy")
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{bar.get_height():.1f}%", ha="center")
    save_figure("heldout_factual_accuracy.png")

    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=category, x="primary_category", y="factual_accuracy", hue="system_name",
        hue_order=order, palette=PALETTE,
    )
    plt.ylim(0, 1)
    plt.ylabel("Within-category factual accuracy")
    plt.xlabel("Question category")
    plt.title("Held-out accuracy by category")
    save_figure("heldout_accuracy_by_category.png")

    quality_metrics = ["corpus_support_rate", "partial_support_rate", "hallucination_rate"]
    quality = plot_frame.melt(id_vars="system_name", value_vars=quality_metrics, var_name="metric", value_name="rate")
    plt.figure(figsize=(9, 5))
    sns.barplot(data=quality, x="system_name", y="rate", hue="metric", order=order)
    plt.ylim(0, 1)
    plt.ylabel("Claim-level rate")
    plt.xlabel("System")
    plt.title("Held-out corpus support and hallucination")
    save_figure("heldout_claim_support_hallucination.png")


plot_development_retrieval(development_summary_df)
if COMPLETED_SCORING_WORKBOOK is not None:
    plot_heldout_results(overall_summary_df, category_summary_df)


## 18. Guarded final export and integrity record

The final export is blocked until the full 270-output held-out run, locked researcher scoring, claim-level summaries, primary statistics, RQ3 tables and figures all exist. This prevents a preprocessing-only archive from being mistaken for a completed experiment. The output inventory hashes every included file.


In [ ]:
def create_final_export() -> Path:
    required = [
        DIRS["frozen"] / "freeze_manifest.json",
        DIRS["protocol"] / "protocol_freeze.json",
        DIRS["protocol"] / "scoring_rubric.json",
        DIRS["protocol"] / "requirements_frozen.txt",
        DIRS["development"] / "development_retrieval_runs_with_D6.csv",
        DIRS["runs"] / "heldout_results.csv",
        DIRS["evaluation"] / "researcher_scoring_lock.json",
        DIRS["evaluation"] / "heldout_researcher_scored_unblinded.csv",
        DIRS["evaluation"] / "primary_statistical_tests.csv",
        DIRS["evaluation"] / "heldout_system_summary.csv",
        DIRS["evaluation"] / "rq3_retrieval_accuracy_contingency.csv",
        DIRS["evaluation"] / "rq3_odds_ratios.csv",
        DIRS["evaluation"] / "rq3_mrr_completeness_spearman.csv",
        DIRS["figures"] / "heldout_factual_accuracy.png",
        DIRS["figures"] / "heldout_claim_support_hallucination.png",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Cannot export; required files are missing: " + "; ".join(missing))

    heldout = pd.read_csv(DIRS["runs"] / "heldout_results.csv", dtype=str, keep_default_na=False)
    expected_pairs = set(
        (question_id, system_name)
        for question_id in questions_df.loc[questions_df["split"] == "held-out", "question_id"]
        for system_name in ["LLM_ONLY", "BASIC_RAG", "ENHANCED_RAG"]
    )
    actual_pairs = set(zip(heldout["question_id"], heldout["system_name"]))
    if len(heldout) != 270 or actual_pairs != expected_pairs:
        raise ValueError("Final export requires exactly the expected 270 unique held-out question/system outputs")

    inventory = []
    for label in ["frozen", "development", "protocol", "runs", "evaluation", "figures"]:
        folder = DIRS[label]
        for item in file_inventory(folder):
            inventory.append({"section": label, **item})
    inventory_frame = pd.DataFrame(inventory)
    write_utf8_csv(inventory_frame, DIRS["exports"] / "output_file_inventory.csv")
    run_record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_freeze_sha256": DATASET_FREEZE["freeze_record_sha256"],
        "protocol_sha256": FROZEN_PROTOCOL["protocol_sha256"],
        "files_in_inventory": len(inventory_frame),
        "heldout_outputs": len(heldout),
        "heldout_generation_complete": True,
        "locked_researcher_scoring_present": True,
        "claim_level_evaluation_present": True,
        "rq3_analysis_present": True,
    }
    (DIRS["exports"] / "run_completion_record.json").write_text(
        json.dumps(run_record, indent=2), encoding="utf-8"
    )

    archive_name = f"Reproducibility_Export_{utc_stamp()}"
    external_base = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_{archive_name}"
    temporary_archive = Path(shutil.make_archive(str(external_base), "zip", root_dir=PROJECT_ROOT))
    archive_path = DIRS["exports"] / temporary_archive.name
    if archive_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing export: {archive_path}")
    shutil.move(str(temporary_archive), archive_path)
    print("Export archive:", archive_path)
    print("Export SHA-256:", sha256_file(archive_path))
    return archive_path


if COMPLETED_SCORING_WORKBOOK is not None:
    FINAL_EXPORT_PATH = create_final_export()
else:
    FINAL_EXPORT_PATH = None
    print(
        "Portfolio rerun complete through generation and retrieval.\n"
        "Researcher-scored analysis and the final dissertation export are not regenerated "
        "because the original completed scoring workbook is not part of the public repository.\n"
        "See the committed results/ files for the authoritative frozen experiment results."
    )
